# Experiment 2: Contrastive "Bridge-Mode" Steering (Mechanistic Activation Intervention)

**Goal:** Isolate a *bridge-mode direction* in the model's residual stream and causally amplify it at inference time.

A creative leap = producing a solution that is **non-default yet constraint-satisfying**, typically enabled by a mechanism **B**.

**Supported tasks** (controlled by `TASK_TYPE` flag in Cell 4):
- **`"aut"`** — Alternative Uses Task (object-based creative uses)
- **`"ps"`** — Problem Solving Task (problem-based creative solutions)

## Protocol

1. Construct paired **teacher-forced** sequences for each dataset item:
   - **Same prompt** (system + user message) — identical for both conditions.
   - **Condition D:** prompt + **D completion** (default/obvious uses or solutions — "listing mode").
   - **Condition B:** prompt + **B-only completion** (mechanism description only — type, rule, unlocks, justification — **no uses/solutions**).
   - The contrast is pure: listing-mode vs mechanism-reasoning. No uses/solutions noise in B.
2. Run a full forward pass for each condition and cache the residual-stream activation **averaged over the first W completion tokens** (where the model is processing listing text vs analytical reasoning text).
3. Compute centroids `μ_D` and `μ_B` across examples; per-item text washes out.
4. Derive the **bridge-mode direction**: `v_bridge = μ_B − μ_D`.
5. At inference, give the model the **same prompt** and inject `α · v_bridge` at generated tokens, sweeping `α`.

**Data:** `DLP/dataset/abcd_{aut,ps}.json` (D padded to 8 items each)

## Experiments summary

| # | Experiment | Cells | What it does |
|---|------------|--------|----------------|
| **1** | **Setup** | 1–2 | Imports, `QwenModelLoader`, load model (e.g. Qwen2-7B-Instruct). |
| **2** | **Contrastive pairs** | 4–5 | See **Contrastive pairs (elaborated)** below. |
| **3** | **Activation collection** | 7–8 | `ResidualStreamCache` hooks layers. For each pair, run D and B forwards; average residual stream over first W completion tokens per condition. Train/eval split (e.g. 80/20) for later steering. |
| **4** | **Bridge direction** | 10, 10b, 11 | Per layer: μ_D, μ_B, `v_bridge = μ_B − μ_D`. Optional Cell 10b: PCA-refine v_bridge. Cell 11: pick best layer (e.g. by `rel_signal`, `frac_positive`). |
| **5** | **Steered generation (α-sweep)** | 13–14 | Hook adds α·v_steer at chosen layer. Sweep α (e.g. 0, 0.5, 1, …, 4). Modes: `last_prompt_only`, `all_new_tokens`, `first_k_assistant_tokens`. Eval items only (held out from v_bridge). |
| **6** | **Analysis** | 16–18 | Qualitative: show replies per item × α. Quantitative: reply length, token overlap vs α=0 (e.g. Jaccard). Plots: length vs α, overlap vs α, layer diagnostics. |
| **7** | **Few-shot (mechanism in prompt)** | FS1–FS2 | No steering. Vary 0/1/2/3-shot examples of (mechanism B → creative solution C). Test if mechanism exposure in prompt improves creativity. |
| **8** | **Two-hop (mechanism then solution)** | 2H1–2H2 | **Baseline:** direct solution. **2-hop model:** Hop 1 = model generates mechanism (with few-shot “source system + mapping” prompt); Hop 2 = solution given that mechanism. **2-hop oracle:** Hop 2 with gold B from dataset. Compare to see impact of hop-1 quality. |
| **9** | **Ablation** | 20 | Fixed α; compare injection modes (`last_prompt_only`, `all_new_tokens`, `all`) on reply length and content. |
| **PE** | **Prompt engineering** | PE1–PE2 | Compare on same eval set: (1) baseline with nothing, (2) ABCD framework in prompt, (3) contrastive steering, (4) multi-hop, (5) few-shot. |
| **10** | **Save** | 22 | Save steering vectors, layer diagnostics, α-sweep and ablation results, quantitative metrics to `bridge_steering_outputs_{TASK_TYPE}/`.

### Contrastive pairs (elaborated) — Cells 4 & 5

**ABCD data (what we load)**  
Each item is a dict with:
- **A** — task: object + setting (AUT) or problem statement (PS).
- **D** — list of 8 default/obvious responses (uses or solutions).
- **B** — mechanism: `{ "type", "rule", "unlocks", "justification" }` (PS also has **C** = creative solution text).

Datasets: `abcd_aut.json` (AUT) or `abcd_aiden.json` (PS). One is chosen via **`TASK_TYPE`** in Cell 4.

**What we build (Cell 4)**  
For each item we build one **contrastive pair** with the **same** prompt and two different **teacher-forced** completions:

| Output | Content |
|--------|--------|
| **prompt_msgs** | `[system, user]` — shared by both conditions. System = (empty). User = task text (AUT: object + "Give 8 unconventional uses…"; PS: problem + "Propose one creative solution…"). |
| **d_completion** | **D condition (listing):** AUT = bullet list of 8 items from `D`. PS = short conventional paragraph built from first 3 entries in `D`. |
| **b_completion** | **B condition (mechanism):** AUT = mechanism only: `Mechanism type: … Rule: … This unlocks: … Justification: …` (no uses). PS = mechanism **plus** creative solution: `Mechanism type: … Rule: …` then full **C** text. |

So: **one prompt**, two completions. Activations later compare “model reading listing” vs “model reading mechanism (and for PS, mechanism→solution).”

**Tokenization and window (Cell 5)**  
- Each pair is tokenized twice: `prompt_msgs + d_completion` → **tok_D**, `prompt_msgs + b_completion` → **tok_B** (chat template applied, then tokenizer).
- From each we get: **full_ids**, **assistant_start** (index of first assistant token), **n_completion_tokens** (length of completion in tokens).
- **WINDOW** = 48 (config constant): we will average residual-stream activations over the **first W** completion tokens.
- **effective_W** = `min(WINDOW, min_comp_D, min_comp_B)` so we never read past the end of the shortest completion.

**Typical stats (printed after Cell 5)**  
- `Tokenization done.  WINDOW = 48, effective W = <value>` (e.g. 46 if shortest completion has 46 tokens).
- `Prompt length (shared): <n> tokens` (e.g. 73).
- `D completion tokens:  min=<m>, example=<e>`.
- `B completion tokens:  min=<m>, example=<e>`.
- `assistant_start (shared): <n>`.

So: **what** = one shared prompt, D vs B teacher-forced completions; **what we get** = per-pair token counts and a single **effective_W** used for activation averaging in the next steps.

In [ ]:
# DLP package (run from DLP/ or notebooks/)
import sys
from pathlib import Path
_root = Path.cwd()
for _ in range(10):
    if (_root / "dlp").is_dir():
        sys.path.insert(0, str(_root))
        break
    _root = _root.parent
from dlp.models import HFLoader
QwenModelLoader = HFLoader  # alias for rest of notebook


In [1]:
"""Cell 1: Imports and QwenModelLoader (from debug.ipynb)."""

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import torch
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


# QwenModelLoader from dlp.models (see previous cell)
print("Imports OK.")


/worxpace/repos/py/nss_networks/DLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK.


In [2]:
"""Cell 2: Load model."""
MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

loader = QwenModelLoader(MODEL_NAME)
loader.load()

model = loader.model
tokenizer = loader.tokenizer

# Inspect layer structure
n_layers = len(model.model.layers)
d_model = model.config.hidden_size
print(f"Model: {MODEL_NAME}")
print(f"Layers: {n_layers}, hidden_size: {d_model}")
print(f"Device: {next(model.parameters()).device}")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.46it/s]


Model: Qwen/Qwen2-7B-Instruct
Layers: 28, hidden_size: 3584
Device: cuda:0


## 1. Load dataset and build teacher-forced contrastive pairs

Set **`TASK_TYPE`** in the next cell to `"aut"` (Alternative Uses) or `"ps"` (Problem Solving).

For each ABCD item we build two **full chat sequences** sharing the **same prompt**:

- **D-condition** (listing): `[system] + [user: task prompt] → [assistant: "- D₁\n- D₂\n..."]`
- **B-condition** (mechanism): `[system] + [user: task prompt] → [assistant: "Mechanism type: ...\nRule: ...\nUnlocks: ...\nJustification: ..."]`

The B completion is **pure mechanism reasoning** — no uses/solutions at all. This gives a clean contrast between "listing defaults" and "analyzing mechanisms."

In [3]:
"""Cell 4: Load dataset and build teacher-forced contrastive pairs.

Design — isolate the B (mechanism reasoning) direction:
  Both conditions share the EXACT SAME prompt (system + user).
  The contrast comes from teacher-forced COMPLETIONS:
    - D completion:  list of default/obvious uses or solutions  (listing mode)
    - B completion:  ONLY the mechanism description (analytical reasoning mode)
                     — type, rule, unlocks, justification — NO uses/solutions at all.

  This ensures v_bridge = μ_B − μ_D captures the difference between
  "listing things" and "analyzing properties/mechanisms", not between
  two different lists.

Supports two task types via TASK_TYPE flag:
  - "aut": Alternative Uses Task  (abcd_aut.json)
  - "ps":  Problem Solving Task   (abcd_ps.json)
"""

import re
import json
from datetime import datetime
from enum import Enum
# ═══════════════════════════════════════════════════════════════════
# TASK_TYPE flag — set to "aut" or "ps" to switch between tasks
# ═══════════════════════════════════════════════════════════════════
class TaskType(Enum):   
    AUT = "aut"
    PS = "ps"

TASK_TYPE = TaskType.AUT  # <── change to "aut" for Alternative Uses Task
assert TASK_TYPE in (TaskType.AUT, TaskType.PS), f"TASK_TYPE must be 'aut' or 'ps', got '{TASK_TYPE}'"
dataset_path = {
    TaskType.PS: '../dataset/abcd_aiden.json',
    TaskType.AUT: '../dataset/abcd_aut.json',
}
DATASET_PATH = dataset_path[TASK_TYPE]
with open(DATASET_PATH) as f:
    abcd_data = json.load(f)

print(f"Task type: {TASK_TYPE.value.upper()}")
print(f"Dataset:   {DATASET_PATH}")
print(f"Loaded {len(abcd_data)} ABCD items.")

OUTPUT_DIR = Path("../results/bridge_steering") / f"{TASK_TYPE.value}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = Path("../results/bridge_steering/bridge_steering_outputs_aut_20260223_072806")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

# ─── helpers ───

def _parse_object_setting(item: dict) -> tuple[str, str]:
    """Extract object and setting from AUT item; use keys if present else parse from A."""
    if "object" in item and "setting" in item:
        return item["object"], item["setting"]
    a = item["A"]
    m = re.search(r" for (?:a |an )?(.+?) (?:in |focusing)", a)
    obj = m.group(1).strip() if m else "object"
    m2 = re.search(r" in (?:an? )?(.+?)(?:\.|,)", a)
    setting = m2.group(1).strip() if m2 else "general"
    return obj, setting


# ─── Single shared system prompt (neutral, empty) ───

SYSTEM_PROMPT = ""


# ─── AUT user prompt ───

def _format_user_prompt_aut(item: dict) -> str:
    """User prompt for Alternative Uses Task (AUT). Aligned with SFT type-3 format."""
    task = item["A"]
    return f"{task}\nReturn exactly 8 unconventional uses.\nFormat: one use per line, starting with \"-\"."


# ─── PS user prompt ───

def _format_user_prompt_ps(item: dict) -> str:
    """User prompt for Problem Solving (PS) task. Canonical (Option A): same for all baselines."""
    problem = item["A"]
    return (
        "Problem Solving Task\n\n"
        f"Problem: {problem}\n\n"
        "Give one creative, non-obvious solution in a short paragraph."
    )

def _format_user_prompt(item: dict) -> str:
    """Route to the correct prompt formatter based on TASK_TYPE."""
    if TASK_TYPE == "ps":
        return _format_user_prompt_ps(item)
    return _format_user_prompt_aut(item)


def _format_D_completion(item: dict) -> str:
    """Default-listing completion (teacher-forced).

    AUT: bullet list of 8 default uses.
    PS:  a conventional, surface-level solution paragraph built from the
         first few D items — matches the single-solution prompt format.
    """
    if TASK_TYPE == "ps":
        # Combine first 3 default approaches into a conventional paragraph
        top = item["D"][:3]
        return (
            f"A straightforward approach: {top[0].lower()}. "
            f"Additionally, {top[1].lower()}. "
            f"To reinforce this, {top[2].lower()}. "
            f"These standard practices should help address the issue."
        )
    return "\n".join(f"- {use}" for use in item["D"])


def _format_B_completion_aut(item: dict) -> str:
    """Mechanism-only completion for AUT (teacher-forced).

    Pure analytical reasoning — type, rule, unlocks, justification.
    NO uses list.  This is the "mechanism detection" mode we want to isolate.
    """
    b = item["B"]
    return (
        f"Mechanism type: {b['type']}\n"
        f"Rule: {b['rule']}\n"
        f"This unlocks: {b['unlocks']}\n"
        # f"Justification: {b['justification']}"
    )


def _format_B_completion_ps(item: dict) -> str:
    """Mechanism + creative-solution completion for PS (teacher-forced).

    Full creative pipeline: mechanism identification → creative application.
    The first ~30 tokens are mechanism reasoning (type + rule),
    then the creative solution C follows naturally.
    With WINDOW ≈ 48 the activation window captures the transition
    from mechanism-finding to creative output — exactly the 'bridge'.
    v_bridge = mean(mechanism→creative) - mean(conventional)
    encodes 'think mechanistically then apply creatively'.
    Item-specific content washes out over 19+ items.
    """
    b = item["B"]
    c = item["C"]
    return (
        f"Mechanism type: {b['type']}\n"
        f"Rule: {b['rule']}\n\n"
        f"{c}"
    )


def _format_B_completion(item: dict) -> str:
    """Route to the correct B-completion formatter based on TASK_TYPE."""
    if TASK_TYPE == "ps":
        return _format_B_completion_ps(item)
    return _format_B_completion_aut(item)


def build_contrastive_pair(item: dict) -> dict:
    """Build a contrastive pair: same prompt, different teacher-forced completions.

    D: prompt + conventional-solution completion  (default mode)
    B: prompt + creative-solution completion       (creative mode)

    v_bridge = mean(acts_B) - mean(acts_D) at early completion tokens
    captures the creative-thinking direction (item content averages out).
    """
    user_prompt = _format_user_prompt(item)

    prompt_msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    d_completion = _format_D_completion(item)
    b_completion = _format_B_completion(item)

    return {
        "id": item.get("id", ""),
        "domain": item.get("domain", item.get("task", "")),
        "user_prompt": user_prompt,
        "prompt_msgs": prompt_msgs,
        "d_completion": d_completion,
        "b_completion": b_completion,
    }


pairs = [build_contrastive_pair(item) for item in abcd_data]

print(f"Built {len(pairs)} contrastive pairs  [TASK_TYPE = {TASK_TYPE!r}].")
print(f"\nSystem: {repr(SYSTEM_PROMPT)}")
print(f"\nUser prompt (first 3 lines):")
print("  " + "\n  ".join(pairs[0]["user_prompt"].split("\n")[:3]))
print(f"\nD completion (first item):")
print("  " + "\n  ".join(pairs[0]["d_completion"].split("\n")[:4]))
print(f"\nB completion (first item — mechanism only):")
print("  " + "\n  ".join(pairs[0]["b_completion"].split("\n")))


Task type: AUT
Dataset:   ../DLP/dataset/abcd_aut.json
Loaded 150 ABCD items.
Output dir: bridge_steering_outputs_aut_20260223_072806
Built 150 contrastive pairs  [TASK_TYPE = <TaskType.AUT: 'aut'>].

System: ''

User prompt (first 3 lines):
  Give 8 unconventional uses for a rubber band in an office setting.
  Return exactly 8 unconventional uses.
  Format: one use per line, starting with "-".

D completion (first item):
  - Bundle papers together
  - Organize pens on a desk
  - Seal a bag of snacks
  - Hold a rolled-up poster

B completion (first item — mechanism only):
  Mechanism type: Re-representation
  Rule: Treat it as a controllable tension+friction loop (temporary fastening + force feedback).
  This unlocks: Temporary fixturing, slip-prevention, and tension-based signaling beyond bundling.
  


## 2. Tokenize full sequences and cache activations

We tokenize each full sequence (prompt + teacher-forced completion), find where the
assistant content begins, and define a **window** of the first `W` completion tokens.

We then hook into `model.model.layers[L]`, run a full forward pass, and read the
residual-stream activation **averaged over the W-token window** at the start of
each completion. This captures the model's behavioral mode (default-listing vs
mechanism-reasoning) at the positions where it actually diverges.

- **Layer selection:** Several candidate layers (mid-to-late) are probed.
- **Token position:** First `W` assistant-content tokens (not the last prompt token).

In [4]:
"""Cell 5: Tokenize full sequences (prompt + teacher-forced completion).

For each condition we build the full chat sequence and find where the
assistant content begins.  Activations will be read at positions
[assistant_start, assistant_start + W) — the first W completion tokens.
"""

WINDOW = 48  # number of early completion tokens to average over


def tokenize_full_sequence(prompt_msgs: list[dict], completion: str, tokenizer) -> dict:
    """Tokenize prompt + teacher-forced completion.

    Returns dict with full_ids, assistant_start, and completion token count.
    """
    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=False, add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, return_tensors="pt")["input_ids"]
    assistant_start = prompt_ids.shape[1]

    full_msgs = prompt_msgs + [{"role": "assistant", "content": completion}]
    full_text = tokenizer.apply_chat_template(
        full_msgs, tokenize=False, add_generation_prompt=False
    )
    full_ids = tokenizer(full_text, return_tensors="pt")["input_ids"]

    return {
        "full_ids": full_ids,
        "prompt_ids": prompt_ids,
        "assistant_start": assistant_start,
        "last_prompt_pos": assistant_start - 1,
        "n_completion_tokens": full_ids.shape[1] - assistant_start,
    }


for p in pairs:
    p["tok_D"] = tokenize_full_sequence(p["prompt_msgs"], p["d_completion"], tokenizer)
    p["tok_B"] = tokenize_full_sequence(p["prompt_msgs"], p["b_completion"], tokenizer)

min_comp_D = min(p["tok_D"]["n_completion_tokens"] for p in pairs)
min_comp_B = min(p["tok_B"]["n_completion_tokens"] for p in pairs)
effective_W = min(WINDOW, min_comp_D, min_comp_B)

print(f"Tokenization done.  WINDOW = {WINDOW}, effective W = {effective_W}")
print(f"  Prompt length (shared): {pairs[0]['tok_D']['assistant_start']} tokens")
print(f"  D completion tokens:  min={min_comp_D}, example={pairs[0]['tok_D']['n_completion_tokens']}")
print(f"  B completion tokens:  min={min_comp_B}, example={pairs[0]['tok_B']['n_completion_tokens']}")
print(f"  assistant_start (shared): {pairs[0]['tok_D']['assistant_start']}")

Tokenization done.  WINDOW = 48, effective W = 35
  Prompt length (shared): 45 tokens
  D completion tokens:  min=35, example=56
  B completion tokens:  min=37, example=51
  assistant_start (shared): 45


### Activation caching infrastructure

`ResidualStreamCache` hooks into transformer layers and captures the full
residual-stream tensor for a forward pass. We then read activations at
the **first W completion-token positions** (not last-prompt-token).

In [5]:
"""Cell 7: Activation caching infrastructure."""

class ResidualStreamCache:
    """
    Register forward hooks on transformer layers to capture residual-stream
    activations at specified token positions.
    """

    def __init__(self, model, layers: list[int]):
        """
        Args:
            model: The HF CausalLM model (model.model.layers is the layer list).
            layers: Which layer indices to hook.
        """
        self.model = model
        self.layers = layers
        self._hooks = []
        self._cache: dict[int, torch.Tensor] = {}  # layer_idx -> (batch, seq, d_model)

    def _make_hook(self, layer_idx: int):
        def hook_fn(module, input, output):
            # output is (hidden_states, ...) or just hidden_states depending on config
            if isinstance(output, tuple):
                hidden = output[0]
            else:
                hidden = output
            self._cache[layer_idx] = hidden.detach().cpu()
        return hook_fn

    def register(self):
        """Register hooks."""
        self.remove()
        for L in self.layers:
            h = self.model.model.layers[L].register_forward_hook(self._make_hook(L))
            self._hooks.append(h)
        return self

    def remove(self):
        """Remove all hooks."""
        for h in self._hooks:
            h.remove()
        self._hooks.clear()
        self._cache.clear()

    def get(self, layer_idx: int, token_pos: int) -> torch.Tensor:
        """
        Get the cached activation for a specific layer and token position.
        Returns shape (d_model,).
        """
        return self._cache[layer_idx][0, token_pos, :]

    @torch.no_grad()
    def forward_and_cache(self, input_ids: torch.Tensor) -> None:
        """
        Run a forward pass (no generation, just one pass) to populate the cache.
        """
        self._cache.clear()
        input_ids = input_ids.to(next(self.model.parameters()).device)
        self.model(input_ids=input_ids, use_cache=False)


# Choose layers to probe: spread across mid-to-late range
PROBE_LAYERS = list(range(n_layers // 4, n_layers, 2))
# Ensure we have at least layers at 1/4, 1/2, 3/4
# PROBE_LAYERS = sorted(set(PROBE_LAYERS + [n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers - 1]))
# PROBE_LAYERS = [l for l in PROBE_LAYERS if 0 <= l < n_layers]
# PROBE_LAYERS = [20]
print(f"Probing layers: {PROBE_LAYERS}")
print(f"  (out of {n_layers} total layers)")

Probing layers: [7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27]
  (out of 28 total layers)


In [6]:
"""Cell 8: Collect activations at early completion tokens (teacher-forced).

For each condition we run a FULL forward pass (prompt + completion) and
read activations averaged over the first W completion-token positions.

  D tokens:  model is processing "- Bundle papers together, - Organize pens..."
  B tokens:  model is processing "Mechanism type: Re-representation, Rule: ..."

The activation difference captures listing-mode vs mechanism-reasoning mode.
"""

acts_D = {L: [] for L in PROBE_LAYERS}
acts_B = {L: [] for L in PROBE_LAYERS}

W = min(WINDOW, min_comp_D, min_comp_B)  # effective window from Cell 5

cache = ResidualStreamCache(model, PROBE_LAYERS)
cache.register()

# ─── Use only train_pairs for computing v_bridge ───
# eval items are held out and never influence the direction.
# Change EVAL_SEED to hold out a different random set; use 35 eval (same as sft_bridge_internalization).
import random as _rng
N_EVAL_HOLDOUT = min(40, len(pairs))
EVAL_SEED = 7  # same seed as sft_bridge_internalization for same 35 examples
_eval_idxs = set(_rng.Random(EVAL_SEED).sample(range(len(pairs)), N_EVAL_HOLDOUT))
train_pairs = [p for i, p in enumerate(pairs) if i not in _eval_idxs]
print(f"Caching activations for {len(train_pairs)} train items")
print(f"  Held-out eval indices (seed={EVAL_SEED}): {sorted(_eval_idxs)}")

for p in tqdm(train_pairs, desc="Caching activations"):
    # ── D condition (full forward pass with teacher-forced D completion) ──
    cache.forward_and_cache(p["tok_D"]["full_ids"])
    start_D = p["tok_D"]["assistant_start"]
    for L in PROBE_LAYERS:
        vecs = torch.stack([cache.get(L, t) for t in range(start_D, start_D + W)])
        acts_D[L].append(vecs.mean(dim=0))

    # ── B condition (full forward pass with teacher-forced B-only completion) ──
    cache.forward_and_cache(p["tok_B"]["full_ids"])
    start_B = p["tok_B"]["assistant_start"]
    for L in PROBE_LAYERS:
        vecs = torch.stack([cache.get(L, t) for t in range(start_B, start_B + W)])
        acts_B[L].append(vecs.mean(dim=0))

cache.remove()

print(f"Collected {len(train_pairs)} activation vectors per condition per layer.")
print(f"  Window W = {W} completion tokens averaged per item.")
print(f"  Shape per vector: {acts_D[PROBE_LAYERS[0]][0].shape}")

Caching activations for 110 train items
  Held-out eval indices (seed=7): [5, 6, 9, 12, 14, 15, 17, 18, 22, 23, 24, 28, 31, 37, 38, 39, 50, 53, 54, 57, 61, 69, 71, 73, 74, 82, 93, 101, 107, 108, 109, 111, 120, 126, 127, 129, 130, 134, 137, 145]


Caching activations: 100%|██████████| 110/110 [00:09<00:00, 11.89it/s]

Collected 110 activation vectors per condition per layer.
  Window W = 35 completion tokens averaged per item.
  Shape per vector: torch.Size([3584])


## 3. Compute centroids and the bridge-mode direction

For each probed layer `L`:
- `μ_D[L]` = mean of D-condition activations (listing mode)
- `μ_B[L]` = mean of B-condition activations (mechanism-reasoning mode)
- `v_bridge[L]` = `μ_B[L] − μ_D[L]`

We also compute the cosine similarity between the two centroids and the norm of
`v_bridge` to see which layers show the most separation.

In [7]:
"""Cell 10: Compute centroids, v_bridge, and layer diagnostics."""

centroids_D = {}
centroids_B = {}
v_bridge = {}
layer_stats = []

for L in PROBE_LAYERS:
    stack_D = torch.stack(acts_D[L])    # (N, d_model)
    stack_B = torch.stack(acts_B[L])    # (N, d_model)

    mu_D = stack_D.mean(dim=0)
    mu_B = stack_B.mean(dim=0)
    v = mu_B - mu_D

    centroids_D[L] = mu_D
    centroids_B[L] = mu_B
    v_bridge[L] = v

    # Diagnostics
    cos_sim = torch.nn.functional.cosine_similarity(mu_D.unsqueeze(0), mu_B.unsqueeze(0)).item()
    v_norm = v.norm().item()
    mu_D_norm = mu_D.norm().item()
    mu_B_norm = mu_B.norm().item()
    # Per-item separation: project each pair's difference onto v_bridge direction
    diffs = (stack_B - stack_D).float()  # (N, d_model); float() for .numpy() (BFloat16 unsupported)
    v_unit = (v / (v.norm() + 1e-8)).float()
    projs = (diffs @ v_unit).numpy()  # (N,) projections onto bridge direction

    rel_signal = v_norm / (mu_D_norm + 1e-8)  # |v_bridge| / |μ_D|
    cv = float(projs.std()) / (float(projs.mean()) + 1e-8)  # coefficient of variation

    layer_stats.append({
        "layer": L,
        "cos_centroids": round(cos_sim, 4),
        "|v_bridge|": round(v_norm, 2),
        "|μ_D|": round(mu_D_norm, 2),
        "|μ_B|": round(mu_B_norm, 2),
        "rel_signal": round(rel_signal, 4),
        "CV": round(cv, 4),
        "proj_mean": round(float(projs.mean()), 2),
        "proj_std": round(float(projs.std()), 2),
        "proj_min": round(float(projs.min()), 2),
        "frac_positive": round(float((projs > 0).mean()), 3),
    })

import pandas as pd
print(layer_stats)
stats_df = pd.DataFrame(layer_stats)
print("Layer diagnostics:")
print(stats_df.to_string(index=False))
print(f"\nBest layer by |v_bridge|: {stats_df.loc[stats_df['|v_bridge|'].idxmax(), 'layer']}")
print(f"Best layer by rel_signal: {stats_df.loc[stats_df['rel_signal'].idxmax(), 'layer']}")
print(f"Best layer by frac_positive: {stats_df.loc[stats_df['frac_positive'].idxmax(), 'layer']}")

[{'layer': 7, 'cos_centroids': 0.8125, '|v_bridge|': 11.75, '|μ_D|': 19.5, '|μ_B|': 17.75, 'rel_signal': 0.6026, 'CV': 0.0809, 'proj_mean': 11.71, 'proj_std': 0.95, 'proj_min': 9.27, 'frac_positive': 1.0}, {'layer': 9, 'cos_centroids': 0.832, '|v_bridge|': 14.25, '|μ_D|': 25.5, '|μ_B|': 21.88, 'rel_signal': 0.5588, 'CV': 0.094, 'proj_mean': 14.25, 'proj_std': 1.34, 'proj_min': 11.11, 'frac_positive': 1.0}, {'layer': 11, 'cos_centroids': 0.8164, '|v_bridge|': 16.38, '|μ_D|': 28.12, '|μ_B|': 24.0, 'rel_signal': 0.5822, 'CV': 0.0949, 'proj_mean': 16.31, 'proj_std': 1.55, 'proj_min': 12.93, 'frac_positive': 1.0}, {'layer': 13, 'cos_centroids': 0.8047, '|v_bridge|': 18.38, '|μ_D|': 30.62, '|μ_B|': 26.75, 'rel_signal': 0.6, 'CV': 0.1017, 'proj_mean': 18.3, 'proj_std': 1.86, 'proj_min': 14.08, 'frac_positive': 1.0}, {'layer': 15, 'cos_centroids': 0.8203, '|v_bridge|': 21.38, '|μ_D|': 37.0, '|μ_B|': 30.88, 'rel_signal': 0.5777, 'CV': 0.1049, 'proj_mean': 21.45, 'proj_std': 2.25, 'proj_min': 16

In [8]:
"""Cell 10b (OPTIONAL): PCA-refined v_bridge.

The raw centroid difference mu_B - mu_D is noisy with only 20 items --
it mixes the consistent "mechanism-reasoning mode" direction with
item-specific text encoding and entangled format/language directions.

PCA on the per-item differences {act_B_i - act_D_i} extracts the
direction that explains the MOST variance across items.  If the
mechanism-mode direction is the dominant signal, PC1 will be it,
cleaned of item-specific noise.

Set USE_PCA = True to replace v_bridge with the PCA-refined version.
"""

USE_PCA = True

if USE_PCA:
    v_bridge_raw = {L: v_bridge[L].clone() for L in PROBE_LAYERS}
    pca_info = {}

    for L in PROBE_LAYERS:
        stack_D = torch.stack(acts_D[L]).float()   # (N, d_model)
        stack_B = torch.stack(acts_B[L]).float()   # (N, d_model)
        diffs = stack_B - stack_D                   # (N, d_model)

        # Center the diffs (PCA convention)
        diffs_centered = diffs - diffs.mean(dim=0)

        # SVD to get principal components
        U, S, Vt = torch.linalg.svd(diffs_centered, full_matrices=False)
        # Vt[0] is PC1 direction (unit vector)
        pc1 = Vt[0]

        # Ensure PC1 points in the same direction as the mean difference
        # (SVD sign is arbitrary)
        mean_diff = diffs.mean(dim=0)
        if (pc1 @ mean_diff) < 0:
            pc1 = -pc1

        # Scale PC1 to have the same norm as the raw v_bridge
        # so alpha values remain comparable
        raw_norm = v_bridge[L].float().norm()
        v_pca = pc1 * raw_norm

        # Variance explained by PC1
        total_var = (S ** 2).sum().item()
        pc1_var = (S[0] ** 2).item()
        var_explained = pc1_var / total_var if total_var > 0 else 0.0

        # Cosine similarity between raw v_bridge and PCA v_bridge
        cos_raw_pca = torch.nn.functional.cosine_similarity(
            mean_diff.unsqueeze(0), pc1.unsqueeze(0)
        ).item()

        pca_info[L] = {
            "var_explained_pc1": round(var_explained, 4),
            "cos_raw_vs_pca": round(cos_raw_pca, 4),
            "raw_norm": round(raw_norm.item(), 2),
        }

        # Replace v_bridge with PCA version
        v_bridge[L] = v_pca.to(v_bridge[L].dtype)

    print("PCA refinement applied to v_bridge.")
    print(f"{'Layer':>5}  {'VarExpl PC1':>12}  {'cos(raw,pca)':>13}  {'raw_norm':>9}")
    for L in PROBE_LAYERS:
        info = pca_info[L]
        print(f"{L:>5}  {info['var_explained_pc1']:>12.4f}  {info['cos_raw_vs_pca']:>13.4f}  {info['raw_norm']:>9.2f}")
else:
    print("PCA refinement SKIPPED (USE_PCA = False). Using raw centroid difference.")


PCA refinement applied to v_bridge.
Layer   VarExpl PC1   cos(raw,pca)   raw_norm
    7        0.0753         0.0817      11.73
    9        0.0881         0.1498      14.25
   11        0.0931         0.1681      16.34
   13        0.0950         0.2392      18.33
   15        0.0983         0.3073      21.41
   17        0.1078         0.3148      24.26
   19        0.1165         0.3714      31.47
   21        0.1108         0.2723      44.60
   23        0.1002         0.1580      60.44
   25        0.0951         0.1375      82.83
   27        0.1157         0.5829     208.17


In [10]:
"""Cell 11: Select the best layer for steering."""

# Pick layer with highest relative signal (|v_bridge| / |μ_D|) among layers
# where all items separate correctly (frac_positive == 1.0).
# This avoids always picking the deepest layer just because norms grow with depth.
_candidates = stats_df[stats_df["frac_positive"] >= 1.0]
if _candidates.empty:
    _candidates = stats_df  # fallback: use all layers
best_row = _candidates.sort_values("rel_signal", ascending=False).iloc[0]
STEER_LAYER = int(best_row["layer"])
# STEER_LAYER = 19  # uncomment to override automatic selection
print(f"Selected STEER_LAYER = {STEER_LAYER}")
print(f"  rel_signal = {best_row['rel_signal']}")
print(f"  |v_bridge| = {best_row['|v_bridge|']}")
print(f"  CV = {best_row['CV']}")
print(f"  frac_positive = {best_row['frac_positive']}")

# v_bridge may be raw centroid diff or PCA-refined (see Cell 10b above).
v_steer = v_bridge[STEER_LAYER].clone()
print(f"  v_steer norm = {v_steer.norm().item():.2f}")
if USE_PCA:
    print(f"  (PCA-refined; scaled to match raw norm)")
else:
    print(f"  (raw centroid difference)")


Selected STEER_LAYER = 27
  rel_signal = 1.2606
  |v_bridge| = 208.0
  CV = 0.1434
  frac_positive = 1.0
  v_steer norm = 208.00
  (PCA-refined; scaled to match raw norm)


## 4. Steered generation

At inference we hook into layer `L` and add `α · v_bridge` to the residual stream.

Two injection modes:
- **`last_prompt_only`**: inject only at the last prompt token on the prefill pass.
- **`all_new_tokens`**: inject at every generated token position (targeting the mechanism-production window).

We sweep `α` over a modest range (e.g. 0, 0.5, 1, 2, 3, 4) so the added vector does not overwhelm the residual stream.

In [11]:
"""Cell 13: Steered generation infrastructure (UPDATED: add first_k_assistant_tokens mode)."""

class SteeringHook:
    """
    Hook that adds α * v_steer to the residual stream at layer L during generation.

    Modes:
      - "last_prompt_only": inject only at position last_prompt_pos on the first (prefill) forward pass.
      - "all_new_tokens": inject at every generated token position (decode steps) + last prompt token on prefill.
      - "all": inject at every position on every forward pass.
      - "first_k_assistant_tokens": inject only for the first K decode steps (assistant tokens),
         plus optionally at last prompt token on prefill.
    """

    def __init__(
        self,
        model,
        layer_idx: int,
        v_steer: torch.Tensor,
        alpha: float,
        mode: str = "all_new_tokens",
        last_prompt_pos: int = -1,
        k_assist: int = 16,                 # NEW
        also_inject_last_prompt: bool = True # NEW
    ):
        self.model = model
        self.layer_idx = layer_idx
        self.device = next(model.parameters()).device
        self.v_steer = v_steer.to(self.device).to(next(model.parameters()).dtype)
        self.alpha = float(alpha)
        self.mode = mode
        self.last_prompt_pos = int(last_prompt_pos)
        self.k_assist = int(k_assist)
        self.also_inject_last_prompt = bool(also_inject_last_prompt)

        self._hook = None
        self._decode_step = 0  # NEW: counts assistant tokens generated (decode passes)

    def _hook_fn(self, module, input, output):
        # unpack output
        if isinstance(output, tuple):
            hidden = output[0]
            rest = output[1:]
        else:
            hidden = output
            rest = None

        # prefill vs decode heuristic:
        # prefill has seq_len > 1 (prompt length), decode steps typically have seq_len == 1
        is_prefill = (hidden.shape[1] > 1)

        steering = self.alpha * self.v_steer  # (d_model,)

        if self.mode == "last_prompt_only":
            if is_prefill and self.last_prompt_pos >= 0:
                hidden = hidden.clone()
                hidden[0, self.last_prompt_pos, :] += steering

        elif self.mode == "all_new_tokens":
            hidden = hidden.clone()
            if is_prefill:
                if self.last_prompt_pos >= 0:
                    hidden[0, self.last_prompt_pos, :] += steering
            else:
                hidden[:, :, :] += steering  # (1,1,d_model) each decode step

        elif self.mode == "all":
            hidden = hidden + steering

        elif self.mode == "first_k_assistant_tokens":
            hidden = hidden.clone()
            if is_prefill:
                if self.also_inject_last_prompt and self.last_prompt_pos >= 0:
                    hidden[0, self.last_prompt_pos, :] += steering
            else:
                # decode step -> counts assistant tokens
                self._decode_step += 1
                if self._decode_step <= self.k_assist:
                    hidden[:, :, :] += steering

        else:
            raise ValueError(f"Unknown mode: {self.mode}")

        if rest is not None:
            return (hidden,) + rest
        return hidden

    def register(self):
        self.remove()
        self._decode_step = 0
        self._hook = self.model.model.layers[self.layer_idx].register_forward_hook(self._hook_fn)
        return self

    def remove(self):
        if self._hook is not None:
            self._hook.remove()
            self._hook = None


@torch.no_grad()
def steered_generate(
    loader: QwenModelLoader,
    prompt_msgs: list[dict],
    layer_idx: int,
    v_steer: torch.Tensor,
    alpha: float,
    mode: str = "all_new_tokens",
    max_new_tokens: int = 384,
    do_sample: bool = False,
    temperature: float = 1.0,
    k_assist: int = 16,                    # NEW
    also_inject_last_prompt: bool = True,  # NEW
) -> str:
    """Generate with activation steering."""
    model, tokenizer = loader.load()

    text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    last_prompt_pos = inputs["input_ids"].shape[1] - 1

    hook = SteeringHook(
        model,
        layer_idx=layer_idx,
        v_steer=v_steer,
        alpha=alpha,
        mode=mode,
        last_prompt_pos=last_prompt_pos,
        k_assist=k_assist,
        also_inject_last_prompt=also_inject_last_prompt,
    )
    hook.register()
    try:
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    finally:
        hook.remove()

    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


print("Steering infrastructure ready (with first_k_assistant_tokens).")

Steering infrastructure ready (with first_k_assistant_tokens).


In [17]:
# Cell 14: α-sweep — teacher-forced v_bridge, inject at first K assistant tokens
#
# With teacher-forced completions the v_bridge norm is MUCH larger (text content
# differs dramatically), so we use a gentler alpha range.  We inject only for
# the first K assistant tokens (the "mechanism-production window") to target
# the planning phase without overwhelming later tokens.
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75,2.0,2.25,2.5]
# INJECTION_MODE = "first_k_assistant_tokens"
INJECTION_MODE = "all_new_tokens"
K_ASSIST = 48  # inject for first 48 generated tokens (matches WINDOW)

MAX_NEW_TOKENS = 384
DO_SAMPLE = False
TEMPERATURE = 1.

# ─── Train/eval split: reuse the same EVAL_SEED split from Cell 8 ───
# This ensures the eval items here are exactly the ones excluded from v_bridge.
eval_pairs = [p for i, p in enumerate(pairs) if i in _eval_idxs]
train_pairs = [p for i, p in enumerate(pairs) if i not in _eval_idxs]
print(f"Train: {len(train_pairs)} items (used for v_bridge),  Eval: {len(eval_pairs)} items (held out)")
print(f"  Eval indices (seed={EVAL_SEED}): {sorted(_eval_idxs)}")

results = []
for p in tqdm(eval_pairs, desc="Items"):
    for alpha in ALPHAS:
        reply = steered_generate(
            loader,
            p["prompt_msgs"],
            layer_idx=STEER_LAYER,
            v_steer=v_steer,
            alpha=alpha,
            mode=INJECTION_MODE,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            k_assist=K_ASSIST,
        )
        results.append({
            "id": p["id"],
            "domain": p["domain"],
            "A": p["user_prompt"],
            "alpha": alpha,
            "mode": INJECTION_MODE,
            "layer": STEER_LAYER,
            "reply": reply,
        })

results_df = pd.DataFrame(results)
print(f"Generated {len(results_df)} outputs ({len(eval_pairs)} items × {len(ALPHAS)} alphas).")

# Save immediately
results_df.to_json(OUTPUT_DIR / "alpha_sweep_results.json", orient="records", indent=2)
import re
id_to_item = {item["id"]: item for item in abcd_data}
id_order = results_df[results_df["alpha"] == ALPHAS[0]].sort_values("id")["id"].tolist()
id_to_eval_idx = {uid: i for i, uid in enumerate(id_order)}
def _judge_row(r):
    return {"eval_idx": id_to_eval_idx[r["id"]], "id": r["id"], "method": f"steering_alpha_{r['alpha']}", "user_prompt": r["A"], "reply": r["reply"], "reply_len_words": len(str(r["reply"]).split()), "has_mechanism_line": bool(re.search(r"(?i)mechanism\\s*:", str(r["reply"]))), "ground_truth_B": id_to_item[r["id"]]["B"]["rule"], "ground_truth_C": id_to_item[r["id"]]["C"], "problem_text": id_to_item[r["id"]]["A"]}
judge_cols = ["eval_idx", "id", "method", "user_prompt", "reply", "reply_len_words", "has_mechanism_line", "ground_truth_B", "ground_truth_C", "problem_text"]
steering_for_judge = pd.DataFrame([_judge_row(r) for _, r in results_df.iterrows()])[judge_cols]
steering_for_judge.to_csv(OUTPUT_DIR / "steering_alpha_sweep.csv", index=False)
baseline_steer = steering_for_judge[steering_for_judge["method"] == f"steering_alpha_{ALPHAS[0]}"].copy()
baseline_steer["method"] = "steering_baseline"
baseline_steer.to_csv(OUTPUT_DIR / "steering_baseline.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'alpha_sweep_results.json'}, steering_alpha_sweep.csv, steering_baseline.csv")

Train: 110 items (used for v_bridge),  Eval: 40 items (held out)
  Eval indices (seed=7): [5, 6, 9, 12, 14, 15, 17, 18, 22, 23, 24, 28, 31, 37, 38, 39, 50, 53, 54, 57, 61, 69, 71, 73, 74, 82, 93, 101, 107, 108, 109, 111, 120, 126, 127, 129, 130, 134, 137, 145]


Items: 100%|██████████| 40/40 [20:36<00:00, 30.92s/it]

Generated 440 outputs (40 items × 11 alphas).
Saved: bridge_steering_outputs_aut_20260223_072806/alpha_sweep_results.json, steering_alpha_sweep.csv, steering_baseline.csv


In [20]:
"""Cell 16: Qualitative comparison — display replies for each item across α."""

from IPython.display import display, HTML
import html as _html

_TRUNC = 4000  # max chars per reply

# Inline styles so layout works in any theme; one card per item.
_card_style = (
    "border:2px solid #555; border-radius:8px; margin:20px 0; padding:0; "
    "font-family:system-ui,sans-serif; background:#1e1e1e; color:#e0e0e0;"
)
_header_style = (
    "background:#2d2d2d; padding:12px 16px; border-bottom:2px solid #555; "
    "border-radius:6px 6px 0 0;"
)
_row_style = (
    "border-bottom:1px solid #444; padding:12px 16px; "
    "white-space:pre-wrap; word-break:break-word; line-height:1.5; font-size:14px;"
)
_alpha_style = (
    "vertical-align:top; width:72px; padding:12px 12px 12px 16px; "
    "border-right:2px solid #444; background:#252525; font-weight:bold; font-size:14px; color:#b0b0b0;"
)


parts = []
for item_id in results_df["id"].unique():
    sub = results_df[results_df["id"] == item_id]
    prompt_text = sub.iloc[0]["A"]
    domain = sub.iloc[0]["domain"]

    rows = []
    for _, row in sub.iterrows():
        alpha = row["alpha"]
        reply = row["reply"]
        short = _html.escape(reply[:_TRUNC]) + ("…" if len(reply) > _TRUNC else "")
        rows.append(
            f'<tr style="border-bottom:1px solid #444;">'
            f'<td style="{_alpha_style}">α = {alpha:.1f}</td>'
            f'<td style="{_row_style}">{short}</td></tr>'
        )

    parts.append(
        f'<div style="{_card_style}">'
        f'<div style="{_header_style}">'
        f'<div style="font-size:15px; font-weight:bold; margin-bottom:4px;">{_html.escape(item_id)} — {_html.escape(domain)}</div>'
        f'<div style="font-size:13px; color:#a0a0a0;">Prompt: {_html.escape(prompt_text)}</div>'
        f"</div>"
        f'<table style="width:100%; border-collapse:collapse;">{"".join(rows)}</table>'
        f"</div>"
    )

display(HTML("\n".join(parts)))

NameError: name 'results_df' is not defined

## 8. Few-shot experiment: does mechanism exposure drive creativity?

Instead of steering activations, we test whether **showing the model examples of mechanism B → creative solution C** in the prompt makes it produce more creative outputs.

Compare:
- **0-shot**: just the problem (current baseline)
- **1-shot**: one example of mechanism reasoning → creative solution
- **2-shot**: two examples
- **3-shot**: three examples

If few-shot examples help, that validates the hypothesis that mechanism-finding (B) drives creativity.

In [ ]:
"""Cell FS1: Few-shot experiment — mechanism examples in prompt."""

import random as _rng

# ─── Settings ───
N_SHOTS = [0, 1, 2, 3]  # number of few-shot examples
FS_SEED = 42  # seed for selecting few-shot examples
FS_MAX_NEW_TOKENS = 384
FS_DO_SAMPLE = False
FS_TEMPERATURE = 1.
FS_N_SEEDS = 2  # generate multiple times per item to reduce sampling noise

# ─── Build example pool (from training items only) ───
# Reuse _eval_idxs from cell 8 to avoid data leakage
train_items = [item for i, item in enumerate(abcd_data) if i not in _eval_idxs]
eval_items = [item for i, item in enumerate(abcd_data) if i in _eval_idxs]
print(f"Example pool: {len(train_items)} train items, {len(eval_items)} eval items")


def _format_fewshot_example(item: dict) -> str:
    """Format one training item as a few-shot example."""
    b = item["B"]
    return (
        f"Problem: {item['A']}\n"
        f"Mechanism type: {b['type']}\n"
        f"Rule: {b['rule']}\n"
        f"Creative solution: {item['C']}"
    )


def _build_fewshot_prompt(problem: str, examples: list[dict]) -> str:
    """Build user prompt with few-shot examples.

    If examples is empty, falls back to the standard zero-shot prompt.
    """
    if not examples:
        # Zero-shot: canonical PS prompt (Option A)
        return (
            f"Problem Solving Task\n\n"
            f"Problem: {problem}\n\n"
            f"Give one creative, non-obvious solution in a short paragraph."
        )

    # Few-shot: show mechanism examples, then ask for the same
    parts = ["Problem Solving Task\n"]
    parts.append("Here are examples of creative solutions found by identifying the "
                 "underlying mechanism that makes a non-obvious solution possible:\n")
    for i, ex in enumerate(examples, 1):
        parts.append(f"--- Example {i} ---")
        parts.append(_format_fewshot_example(ex))
        parts.append("")
    parts.append("--- Your turn ---")
    parts.append(f"Problem: {problem}\n\n")
    parts.append("Give one creative, non-obvious solution in a short paragraph.")
    return "\n".join(parts)


# ─── Run few-shot generation ───
fs_results = []
rng = _rng.Random(FS_SEED)

for n_shot in N_SHOTS:
    print(f"\n=== {n_shot}-shot ===")
    for item in tqdm(eval_items, desc=f"{n_shot}-shot"):
        # Pick n_shot random examples from training pool
        examples = rng.sample(train_items, min(n_shot, len(train_items)))
        prompt_text = _build_fewshot_prompt(item["A"], examples)
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text},
        ]
        for seed_i in range(FS_N_SEEDS):
            reply = loader.generate(
                msgs,
                max_new_tokens=FS_MAX_NEW_TOKENS,
                do_sample=FS_DO_SAMPLE,
                temperature=FS_TEMPERATURE,
            )
            fs_results.append({
                "id": item.get("id", ""),
                "domain": item.get("domain", ""),
                "A": item["A"],
                "n_shot": n_shot,
                "seed_i": seed_i,
                "reply": reply,
            })


Example pool: 110 train items, 40 eval items

=== 0-shot ===


0-shot: 100%|██████████| 40/40 [05:11<00:00,  7.79s/it]



=== 1-shot ===


1-shot: 100%|██████████| 40/40 [07:32<00:00, 11.32s/it]



=== 2-shot ===


2-shot: 100%|██████████| 40/40 [07:28<00:00, 11.20s/it]



=== 3-shot ===


3-shot: 100%|██████████| 40/40 [07:11<00:00, 10.79s/it]


NameError: name 'pd' is not defined

In [24]:
"""Cell FS1 (AUT): Few-shot experiment — mechanism examples in prompt (Option 2 framing)."""

import random as _rng

# ─── Settings ───
N_SHOTS = [0, 1, 2, 3]  # number of few-shot examples
FS_SEED = 42  # seed for selecting few-shot examples
FS_MAX_NEW_TOKENS = 384
FS_DO_SAMPLE = False
FS_TEMPERATURE = 1.0
FS_N_SEEDS = 2  # generate multiple times per item to reduce sampling noise

# ─── Build example pool (from training items only) ───
# Reuse _eval_idxs from cell 8 to avoid data leakage
train_items = [item for i, item in enumerate(abcd_data) if i not in _eval_idxs]
eval_items  = [item for i, item in enumerate(abcd_data) if i in _eval_idxs]
print(f"Example pool: {len(train_items)} train items, {len(eval_items)} eval items")


def _format_user_prompt_aut_from_A(a_text: str) -> str:
    """Keep your original AUT prompt format, but accept raw A text."""
    return (
        f"{a_text}\n"
        f"Return exactly 8 unconventional uses.\n"
        f"Format: one use per line, starting with \"-\"."
    )


def _format_fewshot_example_aut(item: dict) -> str:
    """Format one training item as a few-shot example for AUT.

    Expects:
      - item["A"]: AUT task text
      - item["B"]: mechanism dict {type, rule, unlocks, justification}
      - item["C"]: list of 8 creative uses (preferred) OR fallback to item["D"]
    """
    b = item["B"]
    uses = item.get("C", item.get("D", []))
    uses = list(uses)[:8]
    if len(uses) < 8:
        uses = uses + ["(missing use)"] * (8 - len(uses))

    uses_block = "\n".join(f"- {u}" for u in uses)

    return (
        f"{_format_user_prompt_aut_from_A(item['A'])}\n\n"
        f"Mechanism type: {b['type']}\n"
        f"Rule: {b['rule']}\n"
        f"This unlocks: {b.get('unlocks','')}\n"
        f"Justification: {b.get('justification','')}\n\n"
        f"{uses_block}"
    )


def _build_fewshot_prompt_aut(a_text: str, examples: list[dict]) -> str:
    """Build user prompt with few-shot examples for AUT (Option 2 framing)."""
    if not examples:
        # Zero-shot: same as the standard AUT prompt
        return _format_user_prompt_aut_from_A(a_text)

    parts = ["Alternative Uses Task (AUT)\n"]
    parts.append(
        "Think of a mechanism as a *bridge* from a solved structure in another domain into this object.\n"
        "A good mechanism is testable: it states a structural mapping (what corresponds to what) and explains "
        "what it unlocks beyond the default.\n"
        "This is why mechanisms produce better uses: they constrain the search to a new, coherent family "
        "instead of superficial novelty.\n"
    )

    for i, ex in enumerate(examples, 1):
        parts.append(f"--- Example {i} ---")
        parts.append(_format_fewshot_example_aut(ex))
        parts.append("")

    parts.append("--- Your turn ---")
    parts.append(_format_user_prompt_aut_from_A(a_text))
    parts.append(
        "\nBefore listing uses, write a mechanism block:\n"
        "Mechanism type: ...\n"
        "Rule: ...\n"
        "This unlocks: ...\n"
        "Justification: ...\n"
        "Then output exactly 8 unconventional uses as bullet lines (\"-\")."
    )
    return "\n".join(parts)


# ─── Run few-shot generation ───
fs_results = []
rng = _rng.Random(FS_SEED)

for n_shot in N_SHOTS:
    print(f"\n=== {n_shot}-shot (AUT) ===")
    for item in tqdm(eval_items, desc=f"{n_shot}-shot"):
        examples = rng.sample(train_items, min(n_shot, len(train_items)))
        prompt_text = _build_fewshot_prompt_aut(item["A"], examples)
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text},
        ]
        for seed_i in range(FS_N_SEEDS):
            reply = loader.generate(
                msgs,
                max_new_tokens=FS_MAX_NEW_TOKENS,
                do_sample=FS_DO_SAMPLE,
                temperature=FS_TEMPERATURE,
            )
            fs_results.append({
                "id": item.get("id", ""),
                "domain": item.get("domain", ""),
                "A": item["A"],
                "n_shot": n_shot,
                "seed_i": seed_i,
                "reply": reply,
            })

Example pool: 110 train items, 40 eval items

=== 0-shot (AUT) ===


0-shot: 100%|██████████| 40/40 [03:47<00:00,  5.68s/it]



=== 1-shot (AUT) ===


1-shot: 100%|██████████| 40/40 [04:52<00:00,  7.31s/it]



=== 2-shot (AUT) ===


2-shot: 100%|██████████| 40/40 [05:00<00:00,  7.51s/it]



=== 3-shot (AUT) ===


3-shot: 100%|██████████| 40/40 [04:55<00:00,  7.40s/it]


In [25]:
import pandas as pd
fs_df = pd.DataFrame(fs_results)
print(f"\nGenerated {len(fs_df)} outputs "
      f"({len(eval_items)} items × {len(N_SHOTS)} shot-counts × {FS_N_SEEDS} seeds).")
fs_df.to_json(OUTPUT_DIR / "fewshot_results.json", orient="records", indent=2)
fs_df.to_csv(OUTPUT_DIR / "fewshot_results.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'fewshot_results.json'}, fewshot_results.csv")



Generated 320 outputs (40 items × 4 shot-counts × 2 seeds).
Saved: bridge_steering_outputs_aut_20260223_072806/fewshot_results.json, fewshot_results.csv


In [15]:
"""Cell FS2: Display few-shot results side by side."""

for item_id in fs_df["id"].unique():
    sub = fs_df[fs_df["id"] == item_id]
    problem = sub["A"].iloc[0]
    print("=" * 80)
    print(f"{item_id}")
    print(f"Problem: {problem[:120]}...")
    print("=" * 80)
    for n_shot in N_SHOTS:
        shots = sub[sub["n_shot"] == n_shot]
        for _, row in shots.iterrows():
            print(f"\n--- {n_shot}-shot (seed {row['seed_i']}) ---")
            print(row["reply"][:600])
            if len(row["reply"]) > 600:
                print("...")
    print()

# ─── Quick stats ───
print("\n" + "=" * 80)
print("Aggregate stats by n_shot:")
stats = fs_df.groupby("n_shot").agg(
    mean_words=("reply", lambda x: x.str.split().str.len().mean()),
    std_words=("reply", lambda x: x.str.split().str.len().std()),
).round(1)
print(stats)


AUT_009_v2
Problem: Give 8 unusual uses for a sponge focusing on acoustics and vibration....

--- 0-shot (seed 0) ---
Reframing the problem: Instead of looking at sponges as cleaning tools, we can consider their porous structure and flexibility to explore their potential applications in acoustics and vibration control.

Solution: Create acoustic panels using sponges.

Detail: To make acoustic panels, cut sponges into small, uniform pieces and attach them to a backing material such as wood or metal. The sponge's porous structure allows it to absorb sound waves, reducing echo and reverberation in rooms. By varying the size and density of the sponge pieces, you can tailor the absorption properties to specific fr
...

--- 0-shot (seed 1) ---
Reframing the problem: Instead of looking at sponges as cleaning tools, we can consider their porous structure and flexibility to explore their potential applications in acoustics and vibration control.

Solution: Create acoustic panels using sponges.


## 9. Two-hop experiment: mechanism first, then solution

Test whether decomposing the creative process into two explicit steps improves output quality:

**Hop 1 — Find the mechanism:** Ask the model to analyze the problem's properties, find cross-domain analogies, and state a mechanism (type + rule). No solution yet.

**Hop 2 — Apply the mechanism:** Feed the mechanism from hop 1 back as context, then ask the model to generate a creative solution *given that mechanism*.

### Conditions

| Condition | Hop 1 | Hop 2 |
|-----------|-------|-------|
| **Baseline** | — | Direct solution (no mechanism) |
| **2-hop (model B)** | Model generates mechanism | Model generates solution given its own mechanism |
| **Oracle 2-hop (gold B)** | Gold mechanism from dataset | Model generates solution given gold mechanism |

The **oracle** condition tells us the ceiling: if the model *had* a perfect mechanism, how creative would the solution be? The gap between oracle and model-B tells us how much hop-1 quality matters.

In [26]:
"""Cell 2H1 (AUT): Two-hop experiment — mechanism first, then 8 uses."""

import random as _rng2h

# ─── Settings ───
TH_MAX_NEW_TOKENS = 384
TH_DO_SAMPLE = True
TH_TEMPERATURE = 0.7
TH_N_SEEDS = 1
TH_SEED = 42

# Reuse eval/train split from earlier cells
th_eval_items = [item for i, item in enumerate(abcd_data) if i in _eval_idxs]
th_train_items = [item for i, item in enumerate(abcd_data) if i not in _eval_idxs]
print(f"2-hop AUT: {len(th_eval_items)} eval items, {len(th_train_items)} train items")


# ─── Shared AUT prompt format (keep your original style) ───
def _format_user_prompt_aut_from_A(a_text: str) -> str:
    return f'{a_text}\nReturn exactly 8 unconventional uses.\nFormat: one use per line, starting with "-".'


# ─── Hop 1: structured mechanism discovery (NO uses) ───
HOP1_SYSTEM_AUT = (
    "You are an expert at cross-domain analogical transfer. "
    "Your job is to identify a specific, well-understood system from another domain "
    "whose solved structure can be reused to generate unconventional uses of an object. "
    "Do not list any uses. Only output the bridge/mechanism."
)

def _format_hop1_example_aut(item: dict, example_num: int) -> str:
    a, b = item["A"], item["B"]
    source = b.get("justification", b.get("unlocks", b["rule"]))
    mapping = b["rule"]
    return (
        f"Example {example_num}:\n"
        f"Object/setting: {a}\n\n"
        f"1. SOURCE SYSTEM: {source}\n"
        f"2. STRUCTURAL MAPPING: {mapping}\n"
        f"3. MECHANISM (NO USES):\n"
        f"   - Mechanism type: {b['type']}\n"
        f"   - Rule: {b['rule']}\n"
        f"   - This unlocks: {b.get('unlocks','')}\n"
        f"   - Justification: {b.get('justification','')}\n\n"
    )

# Pick two AUT examples from training split (avoid leakage + ensure indices exist)
_rng2h.seed(TH_SEED)
HOP1_EXAMPLE_ITEMS_AUT = _rng2h.sample(th_train_items, k=min(2, len(th_train_items)))

def _build_hop1_prompt_aut(a_text: str, example_items: list | None = None) -> str:
    if example_items is None:
        example_items = HOP1_EXAMPLE_ITEMS_AUT
    examples_block = "---\n".join(
        _format_hop1_example_aut(ex, i + 1) for i, ex in enumerate(example_items[:2])
    )
    return (
        "Here are two examples of the kind of cross-domain bridge I need:\n\n"
        f"{examples_block}"
        "---\n\n"
        "Now do the same for this object:\n\n"
        f"Object/setting: {a_text}\n\n"
        "Instructions:\n"
        "1. SOURCE SYSTEM: Name one concrete, well-known system/algorithm from a DIFFERENT field.\n"
        "2. STRUCTURAL MAPPING: 2-3 sentences mapping parts of the source onto the object.\n"
        "3. MECHANISM (NO USES):\n"
        "   - Mechanism type: [Re-representation, Structural mapping, or Constraint relaxation]\n"
        "   - Rule: [Treat <object facet> as <source-domain element> where <key property>.]\n"
        "   - This unlocks: [What new families of uses become possible]\n"
        "   - Justification: [Why this mapping is valid]\n\n"
        "Be concise. Do NOT list any uses."
    )


# ─── Hop 2: generate 8 uses given mechanism ───
HOP2_SYSTEM_AUT = ""

def _build_hop2_prompt_aut(a_text: str, mechanism_text: str) -> str:
    return (
        f"{_format_user_prompt_aut_from_A(a_text)}\n\n"
        "A mechanism (bridge) was identified:\n"
        f"{mechanism_text}\n\n"
        "Using that mechanism, return exactly 8 unconventional uses.\n"
        "Format: one use per line, starting with \"-\".\n"
        "Do not include anything except the 8 bullet lines."
    )


def _format_gold_mechanism_aut(item: dict) -> str:
    b = item["B"]
    return (
        f"Mechanism type: {b['type']}\n"
        f"Rule: {b['rule']}\n"
        f"This unlocks: {b.get('unlocks','')}\n"
        f"Justification: {b.get('justification','')}"
    ).strip()


# ─── Baseline: direct 8 uses (no mechanism) ───
def _build_baseline_prompt_aut(a_text: str) -> str:
    return _format_user_prompt_aut_from_A(a_text)


# ─── Run the experiment ───
th_results = []

for item in tqdm(th_eval_items, desc="2-hop AUT items"):
    a_text = item["A"]
    item_id = item.get("id", "")
    domain = item.get("domain", "")

    for seed_i in range(TH_N_SEEDS):

        # (1) Baseline (direct uses)
        baseline_msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": _build_baseline_prompt_aut(a_text)},
        ]
        baseline_reply = loader.generate(
            baseline_msgs,
            max_new_tokens=TH_MAX_NEW_TOKENS,
            do_sample=TH_DO_SAMPLE,
            temperature=TH_TEMPERATURE,
        )
        th_results.append({
            "id": item_id, "domain": domain, "A": a_text,
            "condition": "baseline", "seed_i": seed_i,
            "hop1": "", "hop2": baseline_reply,
        })

        # (2) 2-hop with model-generated mechanism
        hop1_msgs = [
            {"role": "system", "content": HOP1_SYSTEM_AUT},
            {"role": "user", "content": _build_hop1_prompt_aut(a_text)},
        ]
        model_mechanism = loader.generate(
            hop1_msgs,
            max_new_tokens=TH_MAX_NEW_TOKENS,
            do_sample=TH_DO_SAMPLE,
            temperature=TH_TEMPERATURE,
        )

        hop2_msgs = [
            {"role": "system", "content": HOP2_SYSTEM_AUT},
            {"role": "user", "content": _build_hop2_prompt_aut(a_text, model_mechanism)},
        ]
        model_uses = loader.generate(
            hop2_msgs,
            max_new_tokens=TH_MAX_NEW_TOKENS,
            do_sample=TH_DO_SAMPLE,
            temperature=TH_TEMPERATURE,
        )
        th_results.append({
            "id": item_id, "domain": domain, "A": a_text,
            "condition": "2hop_model", "seed_i": seed_i,
            "hop1": model_mechanism, "hop2": model_uses,
        })

        # (3) Oracle 2-hop with gold mechanism
        gold_mech = _format_gold_mechanism_aut(item)
        oracle_hop2_msgs = [
            {"role": "system", "content": HOP2_SYSTEM_AUT},
            {"role": "user", "content": _build_hop2_prompt_aut(a_text, gold_mech)},
        ]
        oracle_uses = loader.generate(
            oracle_hop2_msgs,
            max_new_tokens=TH_MAX_NEW_TOKENS,
            do_sample=TH_DO_SAMPLE,
            temperature=TH_TEMPERATURE,
        )
        th_results.append({
            "id": item_id, "domain": domain, "A": a_text,
            "condition": "2hop_oracle", "seed_i": seed_i,
            "hop1": gold_mech, "hop2": oracle_uses,
        })

th_df = pd.DataFrame(th_results)
print(f"\nGenerated {len(th_df)} outputs "
      f"({len(th_eval_items)} items × 3 conditions × {TH_N_SEEDS} seeds).")
th_df.to_json(OUTPUT_DIR / "twohop_results_aut.json", orient="records", indent=2)
th_df.to_csv(OUTPUT_DIR / "twohop_results_aut.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'twohop_results_aut.json'}, twohop_results_aut.csv")

2-hop AUT: 40 eval items, 110 train items


2-hop AUT items: 100%|██████████| 40/40 [08:22<00:00, 12.57s/it]


Generated 120 outputs (40 items × 3 conditions × 1 seeds).
Saved: bridge_steering_outputs_aut_20260223_072806/twohop_results_aut.json, twohop_results_aut.csv


In [16]:
"""Cell 2H1: Two-hop experiment — mechanism first, then solution."""

import random as _rng2h

# ─── Settings ───
TH_MAX_NEW_TOKENS = 384
TH_DO_SAMPLE = True
TH_TEMPERATURE = 0.7
TH_N_SEEDS = 1
TH_SEED = 42

# Reuse eval/train split from earlier cells
th_eval_items = [item for i, item in enumerate(abcd_data) if i in _eval_idxs]
th_train_items = [item for i, item in enumerate(abcd_data) if i not in _eval_idxs]
print(f"2-hop experiment: {len(th_eval_items)} eval items, {len(th_train_items)} train items")


# ─── Hop 1 prompt: structured mechanism discovery ───

HOP1_SYSTEM = (
    "You are an expert at cross-domain analogical transfer. "
    "Your job is to identify a specific, well-understood system from another domain "
    "whose solved structure can be directly reused to address the given problem. "
    "Do not be creative — be precise. Name the concrete source system and "
    "state how its structure maps onto the problem."
)

def _format_hop1_example(item: dict, example_num: int) -> str:
    """Format one ABCD item as a few-shot example (SOURCE SYSTEM, MAPPING, MECHANISM)."""
    a, b = item["A"], item["B"]
    source = b.get("justification", b["rule"])
    mapping = b["rule"]
    return (
        f"Example {example_num}:\n"
        f"Problem: {a}\n\n"
        f"1. SOURCE SYSTEM: {source}\n"
        f"2. STRUCTURAL MAPPING: {mapping}\n"
        f"3. MECHANISM:\n"
        f"   - Mechanism type: {b['type']}\n"
        f"   - Rule: {b['rule']}\n\n"
    )

HOP1_EXAMPLE_INDICES = (2, 0)  # LA traffic (queueing), brick/blanket (thermal RC) from abcd_data

def _build_hop1_prompt(problem: str, example_items: list | None = None) -> str:
    """Ask the model to identify a specific solved system from another domain.
    Few-shot examples come from abcd_data (e.g. abcd_aiden.json) when example_items is None."""
    if example_items is None:
        example_items = [abcd_data[i] for i in HOP1_EXAMPLE_INDICES if i < len(abcd_data)]
    examples_block = "---\n".join(
        _format_hop1_example(ex, i + 1) for i, ex in enumerate(example_items[:2])
    )
    return (
        f"Here are two examples of the kind of cross-domain transfer I need:\n\n"
        f"{examples_block}"
        f"---\n\n"
        f"Now do the same for this problem:\n\n"
        f"Problem: {problem}\n\n"
        f"Instructions:\n"
        f"1. SOURCE SYSTEM: Name one concrete, well-known system from a DIFFERENT "
        f"field. Name the specific system or algorithm, not a vague field.\n"
        f"2. STRUCTURAL MAPPING: In 2-3 sentences, map components of the source "
        f"onto components of this problem.\n"
        f"3. MECHANISM:\n"
        f"   - Mechanism type: [Re-representation, Structural mapping, or "
        f"Constraint relaxation]\n"
        f"   - Rule: [Treat <problem element> as <source-domain element> "
        f"where <key structural property>.]\n\n"
        f"Be concise. Do NOT describe the target domain at length."
    )


# ─── Hop 2 prompt: solution given mechanism ───

HOP2_SYSTEM = ""

def _build_hop2_prompt(problem: str, mechanism_text: str) -> str:
    """Ask the model to generate a creative solution given a mechanism."""
    return (
        f"Problem: {problem}\n\n"
        f"A creative analysis revealed the following mechanism:\n"
        f"{mechanism_text}\n\n"
        f"Using this mechanism as your foundation, propose one creative, "
        f"non-obvious solution.\n"
        f"Describe your solution in detail (3-5 sentences) and explain why it works."
    )


def _format_gold_mechanism(item: dict) -> str:
    """Format the gold (human-written) mechanism from the dataset."""
    b = item["B"]
    return (
        f"Mechanism type: {b['type']}\n"
        f"Rule: {b['rule']}"
    )


# ─── Baseline prompt (direct solution, no mechanism) ───

def _build_baseline_prompt(problem: str) -> str:
    """Same as the standard PS prompt — single direct solution."""
    return (
        f"Problem Solving Task\n\n"
        f"Problem: {problem}\n\n"
        f"Propose one creative, non-obvious solution to this problem.\n"
        f"- First, briefly reframe or re-analyze the problem (1-2 sentences).\n"
        f"- Then describe your solution in detail (3-5 sentences).\n"
        f"- Explain why it works."
    )


# ─── Run the experiment ───

th_results = []

for item in tqdm(th_eval_items, desc="2-hop items"):
    problem = item["A"]
    item_id = item.get("id", "")
    domain = item.get("domain", "")

    for seed_i in range(TH_N_SEEDS):

        # ── Condition 1: Baseline (direct solution, no mechanism) ──
        baseline_msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": _build_baseline_prompt(problem)},
        ]
        baseline_reply = loader.generate(
            baseline_msgs,
            max_new_tokens=TH_MAX_NEW_TOKENS,
            do_sample=TH_DO_SAMPLE,
            temperature=TH_TEMPERATURE,
        )
        th_results.append({
            "id": item_id, "domain": domain, "A": problem,
            "condition": "baseline", "seed_i": seed_i,
            "hop1": "", "hop2": baseline_reply,
        })

        # ── Condition 2: 2-hop with model-generated mechanism ──
        hop1_msgs = [
            {"role": "system", "content": HOP1_SYSTEM},
            {"role": "user", "content": _build_hop1_prompt(problem)},
        ]
        model_mechanism = loader.generate(
            hop1_msgs,
            max_new_tokens=TH_MAX_NEW_TOKENS,
            do_sample=TH_DO_SAMPLE,
            temperature=TH_TEMPERATURE,
        )

        hop2_msgs = [
            {"role": "system", "content": HOP2_SYSTEM},
            {"role": "user", "content": _build_hop2_prompt(problem, model_mechanism)},
        ]
        model_solution = loader.generate(
            hop2_msgs,
            max_new_tokens=TH_MAX_NEW_TOKENS,
            do_sample=TH_DO_SAMPLE,
            temperature=TH_TEMPERATURE,
        )
        th_results.append({
            "id": item_id, "domain": domain, "A": problem,
            "condition": "2hop_model", "seed_i": seed_i,
            "hop1": model_mechanism, "hop2": model_solution,
        })

        # ── Condition 3: Oracle 2-hop with gold mechanism ──
        gold_mech = _format_gold_mechanism(item)
        oracle_hop2_msgs = [
            {"role": "system", "content": HOP2_SYSTEM},
            {"role": "user", "content": _build_hop2_prompt(problem, gold_mech)},
        ]
        oracle_solution = loader.generate(
            oracle_hop2_msgs,
            max_new_tokens=TH_MAX_NEW_TOKENS,
            do_sample=TH_DO_SAMPLE,
            temperature=TH_TEMPERATURE,
        )
        th_results.append({
            "id": item_id, "domain": domain, "A": problem,
            "condition": "2hop_oracle", "seed_i": seed_i,
            "hop1": gold_mech, "hop2": oracle_solution,
        })

th_df = pd.DataFrame(th_results)
print(f"\nGenerated {len(th_df)} outputs "
      f"({len(th_eval_items)} items × 3 conditions × {TH_N_SEEDS} seeds).")
th_df.to_json(OUTPUT_DIR / "twohop_results.json", orient="records", indent=2)
th_df.to_csv(OUTPUT_DIR / "twohop_results.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'twohop_results.json'}, twohop_results.csv")


2-hop experiment: 40 eval items, 110 train items


2-hop items: 100%|██████████| 40/40 [09:17<00:00, 13.94s/it]


Generated 120 outputs (40 items × 3 conditions × 1 seeds).
Saved: bridge_steering_outputs_aut_20260223_072806/twohop_results.json, twohop_results.csv


In [18]:
"""Cell 2H2: Display two-hop results — compare baseline, model 2-hop, and oracle 2-hop."""

CONDITIONS = ["baseline", "2hop_model", "2hop_oracle"]
COND_LABELS = {
    "baseline": "Baseline (direct)",
    "2hop_model": "2-hop (model mechanism)",
    "2hop_oracle": "2-hop (gold mechanism)",
}

for item_id in th_df["id"].unique():
    sub = th_df[th_df["id"] == item_id]
    problem = sub["A"].iloc[0]
    print("=" * 80)
    print(f"{item_id}")
    print(f"Problem: {problem}")

    # Show gold mechanism for reference
    gold_rows = sub[sub["condition"] == "2hop_oracle"]
    if not gold_rows.empty:
        gold_mech_str = gold_rows.iloc[0]["hop1"].replace("\n", "\n    ")
        print(f"\n  GOLD MECHANISM:\n    {gold_mech_str}")
    print("=" * 80)

    for cond in CONDITIONS:
        cond_rows = sub[sub["condition"] == cond]
        for _, row in cond_rows.iterrows():
            print(f"\n--- {COND_LABELS[cond]} (seed {row['seed_i']}) ---")
            if cond == "2hop_model" and row["hop1"]:
                mech_full = row["hop1"].replace("\n", "\n    ")
                print(f"  [MODEL MECHANISM]:\n    {mech_full}")
            print(row["hop2"])
    print()

# ─── Aggregate stats ───
print("\n" + "=" * 80)
print("AGGREGATE STATISTICS")
print("=" * 80)

th_df["reply_len"] = th_df["hop2"].str.len()
th_df["reply_words"] = th_df["hop2"].str.split().str.len()

for cond in CONDITIONS:
    cond_rows = th_df[th_df["condition"] == cond]
    print(f"\n{COND_LABELS[cond]}:")
    print(f"  Mean length (chars): {cond_rows['reply_len'].mean():.0f}")
    print(f"  Mean length (words): {cond_rows['reply_words'].mean():.0f}")

# ─── Show model hop-1 mechanisms for inspection ───
print("\n" + "=" * 80)
print("MODEL-GENERATED MECHANISMS (hop 1) — for qualitative inspection")
print("=" * 80)
model_mechs = th_df[th_df["condition"] == "2hop_model"]
for item_id in model_mechs["id"].unique():
    sub = model_mechs[model_mechs["id"] == item_id]
    print(f"\n{item_id}: {sub['A'].iloc[0]}")
    for _, row in sub.iterrows():
        print(f"  [seed {row['seed_i']}]")
        mech_lines = row["hop1"].strip().split("\n")
        for line in mech_lines:
            print(f"    {line}")
        print()


AUT_009_v2
Problem: Give 8 unusual uses for a sponge focusing on acoustics and vibration.

  GOLD MECHANISM:
    Mechanism type: Re-representation
    Rule: Treat it as a porous damping medium with tunable compression and impedance matching.

--- Baseline (direct) (seed 0) ---
Reframing the Problem: Typically, we think of sponges as tools for cleaning and absorbing liquids. However, considering their unique porous structure and ability to absorb sound, they could potentially be used in various applications related to acoustics and vibration.

Solution: Create an acoustic sculpture using sponges. Design a three-dimensional structure by arranging different sizes and shapes of sponges. This sculpture would serve as a large acoustic diffuser, dispersing sound waves and breaking them into smaller, less disruptive segments.

Explanation: The porous structure of sponges allows them to absorb sound waves. By strategically arranging sponges at different distances and angles within the sculpture

## Prompt engineering comparison (PE1–PE2)

Run **five conditions** on the **same eval set** to compare prompt-only vs steering vs multi-hop vs few-shot:

| Condition | Description |
|-----------|-------------|
| **baseline_nothing** | Minimal prompt: problem + "Give one creative, non-obvious solution in a short paragraph." No framework, no steering. |
| **abcd_framework** | Same problem but prompt introduces the ABCD idea: identify a mechanism (B) that unlocks non-obvious solutions, then give your solution (C). |
| **contrastive_steering** | Same prompt as used for v_bridge (standard PS prompt); activation steering with fixed α (e.g. 1.0). |
| **multi_hop** | Two-hop: Hop 1 = model generates mechanism (hop1 prompt); Hop 2 = solution given that mechanism. |
| **few_shot** | Few-shot prompt with 2 mechanism→solution examples, then ask for the same on the target problem. |

All use the same eval items (held-out from v_bridge). Results are collected in `pe_df` for side-by-side or export to the evaluation notebook.

PE conditions: 100%|██████████| 40/40 [06:11<00:00,  9.28s/it]


Prompt engineering: 80 rows (40 items × 5 conditions)
condition
abcd_framework      40
baseline_nothing    40
dtype: int64
Saved: bridge_steering_outputs_aut_20260223_072806/pe_results.json, pe_results.csv


KeyError: "None of [Index(['eval_idx', 'id', 'method', 'user_prompt', 'reply', 'reply_len_words',\n       'has_mechanism_line', 'ground_truth_B', 'ground_truth_C',\n       'problem_text'],\n      dtype='str')] are in the [columns]"

In [27]:
"""Cell PE1 (AUT): Prompt engineering — run conditions on same eval set (AUT)."""

import random as _rng

# Same eval set as few-shot and 2-hop
pe_eval_items = [item for i, item in enumerate(abcd_data) if i in _eval_idxs]
pe_eval_pairs = [p for i, p in enumerate(pairs) if i in _eval_idxs]
assert len(pe_eval_items) == len(pe_eval_pairs), "eval items and pairs length mismatch"

PE_MAX_NEW_TOKENS = 384
PE_DO_SAMPLE = True
PE_TEMPERATURE = 0.7
PE_STEER_ALPHA = 1.0
PE_FEWSHOT_N = 2
PE_SEED = 42
pe_train_items = [item for i, item in enumerate(abcd_data) if i not in _eval_idxs]
pe_rng = _rng.Random(PE_SEED)


def _format_user_prompt_aut(item: dict) -> str:
    """Original AUT prompt format."""
    task = item["A"]
    return f'{task}\nReturn exactly 8 unconventional uses.\nFormat: one use per line, starting with "-".'


def _pe_prompt_baseline_nothing_aut(item: dict) -> str:
    """Minimal prompt: no framework, no structure (AUT)."""
    return _format_user_prompt_aut(item)


def _pe_prompt_abcd_framework_aut(item: dict) -> str:
    """ABCD framework prompt adapted to AUT (mechanism -> 8 uses)."""
    a = item["A"]
    return (
        "Alternative Uses Task (AUT)\n\n"
        f"**A — Object/setting:**\n{a}\n\n"
        "**ABCD framework:** Creativity comes from *bridges* (mechanisms), not style.\n"
        "- **B** = mechanism: a short, reusable transformation rule that re-represents the object so a "
        "*different family* of uses becomes reachable than the obvious default (**D**).\n"
        "- **C** = your 8 unconventional uses that apply B.\n\n"
        "**Your task:**\n"
        "1) **Identify B (mechanism).** It must: (i) state the transformation, "
        "(ii) name what use-family it unlocks, and (iii) briefly justify why it’s valid.\n"
        "2) **Give C:** return exactly 8 unconventional uses.\n\n"
        "**Output format:**\n"
        "Mechanism type: ...\n"
        "Rule: ...\n"
        "This unlocks: ...\n"
        "Justification: ...\n"
        "Then 8 bullet uses, one per line, starting with \"-\".\n"
    )


def _format_fewshot_example_aut(item: dict) -> str:
    """Few-shot example for AUT: AUT prompt + mechanism + 8 uses."""
    b = item["B"]
    uses = item.get("C", item.get("D", []))
    uses = list(uses)[:8]
    if len(uses) < 8:
        uses = uses + ["(missing use)"] * (8 - len(uses))
    uses_block = "\n".join(f"- {u}" for u in uses)
    return (
        f"{_format_user_prompt_aut(item)}\n\n"
        f"Mechanism type: {b['type']}\n"
        f"Rule: {b['rule']}\n"
        f"This unlocks: {b.get('unlocks','')}\n"
        f"Justification: {b.get('justification','')}\n"
        f"{uses_block}\n"
    )


def _pe_prompt_fewshot_aut(target_item: dict, examples: list[dict]) -> str:
    """Few-shot prompt for AUT (PE_FEWSHOT_N examples)."""
    parts = ["Alternative Uses Task (AUT)\n"]
    parts.append(
        "Mechanisms are *bridges* from solved structures into an object. "
        "They unlock coherent families of unconventional uses beyond defaults.\n"
    )
    for i, ex in enumerate(examples, 1):
        parts.append(f"--- Example {i} ---")
        parts.append(_format_fewshot_example_aut(ex))
    parts.append("--- Your turn ---")
    parts.append(_format_user_prompt_aut(target_item))
    parts.append(
        "\nBefore listing uses, write a mechanism block:\n"
        "Mechanism type: ...\nRule: ...\nThis unlocks: ...\nJustification: ...\n"
        "Then output exactly 8 bullet uses."
    )
    return "\n".join(parts)


# (Optional) If you already adapted 2-hop AUT in another cell, reuse those:
# - HOP1_SYSTEM_AUT, _build_hop1_prompt_aut, HOP2_SYSTEM_AUT, _build_hop2_prompt_aut, _format_gold_mechanism_aut

pe_results = []

for idx, item in enumerate(tqdm(pe_eval_items, desc="PE AUT conditions")):
    item_id = item.get("id", "")
    pair = pe_eval_pairs[idx]

    # (1) baseline_nothing (AUT)
    msgs_baseline = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": _pe_prompt_baseline_nothing_aut(item)},
    ]
    reply_baseline = loader.generate(
        msgs_baseline,
        max_new_tokens=PE_MAX_NEW_TOKENS,
        do_sample=PE_DO_SAMPLE,
        temperature=PE_TEMPERATURE,
    )
    pe_results.append({"id": item_id, "condition": "baseline_nothing", "reply": reply_baseline, "A": item["A"]})

    # (2) abcd_framework (AUT)
    msgs_abcd = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": _pe_prompt_abcd_framework_aut(item)},
    ]
    reply_abcd = loader.generate(
        msgs_abcd,
        max_new_tokens=PE_MAX_NEW_TOKENS,
        do_sample=PE_DO_SAMPLE,
        temperature=PE_TEMPERATURE,
    )
    pe_results.append({"id": item_id, "condition": "abcd_framework", "reply": reply_abcd, "A": item["A"]})

    # (3) contrastive_steering (AUT) — uses the same prompt_msgs as v_bridge pairs
    # reply_steer = steered_generate(
    #     loader, pair["prompt_msgs"],
    #     layer_idx=STEER_LAYER, v_steer=v_steer, alpha=PE_STEER_ALPHA,
    #     mode=INJECTION_MODE, max_new_tokens=PE_MAX_NEW_TOKENS,
    #     do_sample=PE_DO_SAMPLE, temperature=PE_TEMPERATURE,
    # )
    # pe_results.append({"id": item_id, "condition": "contrastive_steering", "reply": reply_steer, "A": item["A"]})

    # (4) multi_hop (AUT) — if your AUT two-hop helpers exist
    # hop1_msgs = [
    #     {"role": "system", "content": HOP1_SYSTEM_AUT},
    #     {"role": "user", "content": _build_hop1_prompt_aut(item["A"])},
    # ]
    # model_mechanism = loader.generate(
    #     hop1_msgs, max_new_tokens=256, do_sample=PE_DO_SAMPLE, temperature=PE_TEMPERATURE,
    # )
    # hop2_msgs = [
    #     {"role": "system", "content": HOP2_SYSTEM_AUT},
    #     {"role": "user", "content": _build_hop2_prompt_aut(item["A"], model_mechanism)},
    # ]
    # reply_multihop = loader.generate(
    #     hop2_msgs, max_new_tokens=PE_MAX_NEW_TOKENS, do_sample=PE_DO_SAMPLE, temperature=PE_TEMPERATURE,
    # )
    # pe_results.append({"id": item_id, "condition": "multi_hop", "reply": reply_multihop, "A": item["A"]})

    # (5) few_shot (AUT)
    # examples = pe_rng.sample(pe_train_items, min(PE_FEWSHOT_N, len(pe_train_items)))
    # prompt_fs = _pe_prompt_fewshot_aut(item, examples)
    # msgs_fs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt_fs}]
    # reply_fewshot = loader.generate(
    #     msgs_fs, max_new_tokens=PE_MAX_NEW_TOKENS, do_sample=PE_DO_SAMPLE, temperature=PE_TEMPERATURE,
    # )
    # pe_results.append({"id": item_id, "condition": "few_shot", "reply": reply_fewshot, "A": item["A"]})

pe_df = pd.DataFrame(pe_results)
print(f"Prompt engineering (AUT): {len(pe_df)} rows")
print(pe_df.groupby("condition").size())

pe_df.to_json(OUTPUT_DIR / "pe_results_aut.json", orient="records", indent=2)
pe_df.to_csv(OUTPUT_DIR / "pe_results_aut.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'pe_results_aut.json'}, pe_results_aut.csv")

PE AUT conditions: 100%|██████████| 40/40 [06:06<00:00,  9.17s/it]

Prompt engineering (AUT): 80 rows
condition
abcd_framework      40
baseline_nothing    40
dtype: int64
Saved: bridge_steering_outputs_aut_20260223_072806/pe_results_aut.json, pe_results_aut.csv


In [ ]:
"""Cell PE1: Prompt engineering — run 5 conditions on same eval set."""

import random as _rng

# Same eval set as few-shot and 2-hop
pe_eval_items = [item for i, item in enumerate(abcd_data) if i in _eval_idxs]
# eval_pairs in same order (by index in pairs)
pe_eval_pairs = [p for i, p in enumerate(pairs) if i in _eval_idxs]
assert len(pe_eval_items) == len(pe_eval_pairs), "eval items and pairs length mismatch"

PE_MAX_NEW_TOKENS = 384
PE_DO_SAMPLE = True
PE_TEMPERATURE = 0.7
PE_STEER_ALPHA = 1.0  # for contrastive_steering
PE_FEWSHOT_N = 2
PE_SEED = 42
pe_train_items = [item for i, item in enumerate(abcd_data) if i not in _eval_idxs]
pe_rng = _rng.Random(PE_SEED)


def _pe_prompt_baseline_nothing(problem: str) -> str:
    """Minimal prompt: no framework, no structure."""
    return (
        f"Problem Solving Task\n\n"
        f"Problem: {problem}\n\n"
        f"Give one creative, non-obvious solution in a short paragraph."
    )


def _pe_prompt_abcd_framework(problem: str) -> str:
    """ABCD framework: A=Situation, B=Mechanism (bridge), C=Creative outcome, D=Default.
    Elicits B as a reusable transformation rule with validity criteria and transformation families."""
    return (
        f"Problem Solving Task\n\n"
        f"**A — Situation:**\n{problem}\n\n"
        "**ABCD framework:** Treat creativity as discovering and applying *bridges*, not just style.\n"
        "- **A** = situation (above). **B** = mechanism: a short, reusable transformation rule that "
        "re-represents or reframes A so a *different family* of solutions becomes reachable than the "
        "obvious default (**D**). **C** = your creative outcome that applies B.\n\n"
        "**Your task:**\n"
        "1. **Identify B (mechanism).** It must: (i) state the transformation, "
        "(ii) name what solution family it unlocks (what becomes possible beyond the default), and "
        "(iii) briefly justify why that solution still satisfies the task and constraints.\n"
        "2. **Give C:** one creative solution in a short paragraph and why it works.\n\n"
        "**Types of B** (you may use any that fit): re-representation (role/function view); "
        "structural mapping (analogy, solve in another domain then map back); intermediate construct "
        "(auxiliary object or subgoal); method shift (invariant, extremal argument, duality, symmetry); "
        "discourse/semantic operator (POV, time, frame, or format change while keeping constraints)."
    )


pe_results = []

for idx, item in enumerate(tqdm(pe_eval_items, desc="PE conditions")):
    problem = item["A"]
    item_id = item.get("id", "")
    pair = pe_eval_pairs[idx]

    # (1) Baseline with nothing
    msgs_baseline = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": _pe_prompt_baseline_nothing(problem)},
    ]
    reply_baseline = loader.generate(
        msgs_baseline, max_new_tokens=PE_MAX_NEW_TOKENS,
        do_sample=PE_DO_SAMPLE, temperature=PE_TEMPERATURE,
    )
    pe_results.append({"id": item_id, "condition": "baseline_nothing", "reply": reply_baseline, "A": problem})

    # (2) ABCD framework in prompt
    msgs_abcd = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": _pe_prompt_abcd_framework(problem)},
    ]
    reply_abcd = loader.generate(
        msgs_abcd, max_new_tokens=PE_MAX_NEW_TOKENS,
        do_sample=PE_DO_SAMPLE, temperature=PE_TEMPERATURE,
    )
    pe_results.append({"id": item_id, "condition": "abcd_framework", "reply": reply_abcd, "A": problem})

    # # (3) Contrastive steering (same prompt as v_bridge, with steering)
    # reply_steer = steered_generate(
    #     loader, pair["prompt_msgs"],
    #     layer_idx=STEER_LAYER, v_steer=v_steer, alpha=PE_STEER_ALPHA,
    #     mode=INJECTION_MODE, max_new_tokens=PE_MAX_NEW_TOKENS,
    #     do_sample=PE_DO_SAMPLE, temperature=PE_TEMPERATURE,
    # )
    # pe_results.append({"id": item_id, "condition": "contrastive_steering", "reply": reply_steer, "A": problem})

    # (4) Multi-hop (hop1 → hop2, model mechanism)
    # hop1_msgs = [
    #     {"role": "system", "content": HOP1_SYSTEM},
    #     {"role": "user", "content": _build_hop1_prompt(problem)},
    # ]
    # model_mechanism = loader.generate(
    #     hop1_msgs, max_new_tokens=256, do_sample=PE_DO_SAMPLE, temperature=PE_TEMPERATURE,
    # )
    # hop2_msgs = [
    #     {"role": "system", "content": HOP2_SYSTEM},
    #     {"role": "user", "content": _build_hop2_prompt(problem, model_mechanism)},
    # ]
    # reply_multihop = loader.generate(
    #     hop2_msgs, max_new_tokens=PE_MAX_NEW_TOKENS, do_sample=PE_DO_SAMPLE, temperature=PE_TEMPERATURE,
    # )
    # pe_results.append({"id": item_id, "condition": "multi_hop", "reply": reply_multihop, "A": problem})

    # (5) Few-shot (PE_FEWSHOT_N mechanism→solution examples)
    # examples = pe_rng.sample(pe_train_items, min(PE_FEWSHOT_N, len(pe_train_items)))
    # prompt_fs = _build_fewshot_prompt(problem, examples)
    # msgs_fs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt_fs}]
    # reply_fewshot = loader.generate(
    #     msgs_fs, max_new_tokens=PE_MAX_NEW_TOKENS, do_sample=PE_DO_SAMPLE, temperature=PE_TEMPERATURE,
    # )
    # pe_results.append({"id": item_id, "condition": "few_shot", "reply": reply_fewshot, "A": problem})

pe_df = pd.DataFrame(pe_results)
print(f"Prompt engineering: {len(pe_df)} rows ({len(pe_eval_items)} items × 5 conditions)")
print(pe_df.groupby("condition").size())
pe_df.to_json(OUTPUT_DIR / "pe_results.json", orient="records", indent=2)
pe_df.to_csv(OUTPUT_DIR / "pe_results.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'pe_results.json'}, pe_results.csv")



PE conditions:  68%|██████▊   | 27/40 [04:26<02:08,  9.90s/it]

In [22]:
# Export judge-friendly CSVs for eval_bridge_internalization (same schema as steering_alpha_sweep.csv)
import re
id_to_item = {item["id"]: item for item in abcd_data}
id_to_eval_idx = {item["id"]: i for i, item in enumerate(pe_eval_items)}
judge_cols = ["eval_idx", "id", "method", "user_prompt", "reply", "reply_len_words", "has_mechanism_line", "ground_truth_B", "ground_truth_C", "problem_text"]

def _pe_judge_row(r, method_name):
    return {
        "eval_idx": id_to_eval_idx[r["id"]], "id": r["id"], "method": method_name,
        "user_prompt": r["A"], "reply": r["reply"],
        "reply_len_words": len(str(r["reply"]).split()),
        "has_mechanism_line": bool(re.search(r"(?i)mechanism\s*:", str(r["reply"]))),
        "ground_truth_B": id_to_item[r["id"]]["B"]["rule"], "ground_truth_C": id_to_item[r["id"]]["C"],
        "problem_text": id_to_item[r["id"]]["A"],
    }

for condition_name, method_name in [
    ("multi_hop", "steering_multi_hop"),
    ("few_shot", "steering_few_shot"),
    ("abcd_framework", "steering_prompt_engineering"),
]:
    sub = pe_df[pe_df["condition"] == condition_name]
    judge_df = pd.DataFrame([_pe_judge_row(r, method_name) for _, r in sub.iterrows()])[judge_cols]
    judge_df.to_csv(OUTPUT_DIR / f"{method_name}.csv", index=False)
print("Saved judge CSVs: steering_multi_hop.csv, steering_few_shot.csv, steering_prompt_engineering.csv")

KeyError: "None of [Index(['eval_idx', 'id', 'method', 'user_prompt', 'reply', 'reply_len_words',\n       'has_mechanism_line', 'ground_truth_B', 'ground_truth_C',\n       'problem_text'],\n      dtype='str')] are in the [columns]"

In [23]:
"""Cell PE2: Display prompt-engineering comparison (one item, all 5 conditions)."""

COND_LABELS = {
    "baseline_nothing": "1. Baseline (nothing)",
    "abcd_framework": "2. ABCD framework in prompt",
    "contrastive_steering": "3. Contrastive steering",
    "multi_hop": "4. Multi-hop",
    "few_shot": "5. Few-shot",
}
# Show first eval item
first_id = pe_df["id"].iloc[0]
sub = pe_df[pe_df["id"] == first_id]
print("=" * 80)
print(f"Item: {first_id}")
print(f"Problem: {sub['A'].iloc[0][:300]}...")
print("=" * 80)
for cond in ["baseline_nothing", "abcd_framework", "contrastive_steering", "multi_hop", "few_shot"]:
    row = sub[sub["condition"] == cond].iloc[0]
    print(f"\n--- {COND_LABELS[cond]} ---\n{row['reply'][:500]}...")
print("\n(Full replies in pe_df.)")

Item: AUT_009_v2
Problem: Give 8 unusual uses for a sponge focusing on acoustics and vibration....

--- 1. Baseline (nothing) ---
One creative, non-obvious use of a sponge in the realm of acoustics and vibration could be as an innovative acoustic treatment for musical instruments, specifically drums. Drum sponges can be crafted into various shapes, sizes, and densities to provide targeted sound absorption or reflection. By placing these sponges around the drum kit, particularly near resonant areas like the drumheads or inside the drum shells, they can help manage unwanted overtones, echoes, or feedback. These custom sponge...

--- 2. ABCD framework in prompt ---
**Situation (A):** Given the situation to brainstorm unusual uses for a sponge focusing on acoustics and vibration, we aim to think creatively about leveraging sponges beyond their typical roles in cleaning and absorbency to explore their potential in enhancing sound and vibration performance.

**Mechanism (B):** **Structural M

IndexError: single positional indexer is out-of-bounds

In [ ]:
"""Cell PE2: Display prompt-engineering comparison (one item, all 5 conditions)."""

COND_LABELS = {
    "baseline_nothing": "1. Baseline (nothing)",
    "abcd_framework": "2. ABCD framework in prompt",
    "contrastive_steering": "3. Contrastive steering",
    "multi_hop": "4. Multi-hop",
    "few_shot": "5. Few-shot",
}
# Show first eval item
first_id = pe_df["id"].iloc[0]
sub = pe_df[pe_df["id"] == first_id]
print("=" * 80)
print(f"Item: {first_id}")
print(f"Problem: {sub['A'].iloc[0][:300]}...")
print("=" * 80)
for cond in ["baseline_nothing", "abcd_framework", "contrastive_steering", "multi_hop", "few_shot"]:
    row = sub[sub["condition"] == cond].iloc[0]
    print(f"\n--- {COND_LABELS[cond]} ---\n{row['reply'][:500]}...")
print("\n(Full replies in pe_df.)")

## 5. Analysis

Compare outputs across `α` values:
- **Qualitative:** Print side-by-side for each item.
- **Quantitative:** Token-level divergence from α=0 baseline; self-BLEU; reply length.
- **Coherence check:** Does the model stay on-task or degenerate at high α?

In [ ]:
"""Cell 16: Qualitative comparison — display replies for each item across α."""

from IPython.display import display, HTML
import html as _html

_TRUNC = 4000  # max chars per reply

# Inline styles so layout works in any theme; one card per item.
_card_style = (
    "border:2px solid #555; border-radius:8px; margin:20px 0; padding:0; "
    "font-family:system-ui,sans-serif; background:#1e1e1e; color:#e0e0e0;"
)
_header_style = (
    "background:#2d2d2d; padding:12px 16px; border-bottom:2px solid #555; "
    "border-radius:6px 6px 0 0;"
)
_row_style = (
    "border-bottom:1px solid #444; padding:12px 16px; "
    "white-space:pre-wrap; word-break:break-word; line-height:1.5; font-size:14px;"
)
_alpha_style = (
    "vertical-align:top; width:72px; padding:12px 12px 12px 16px; "
    "border-right:2px solid #444; background:#252525; font-weight:bold; font-size:14px; color:#b0b0b0;"
)


parts = []
for item_id in results_df["id"].unique():
    sub = results_df[results_df["id"] == item_id]
    prompt_text = sub.iloc[0]["A"]
    domain = sub.iloc[0]["domain"]

    rows = []
    for _, row in sub.iterrows():
        alpha = row["alpha"]
        reply = row["reply"]
        short = _html.escape(reply[:_TRUNC]) + ("…" if len(reply) > _TRUNC else "")
        rows.append(
            f'<tr style="border-bottom:1px solid #444;">'
            f'<td style="{_alpha_style}">α = {alpha:.1f}</td>'
            f'<td style="{_row_style}">{short}</td></tr>'
        )

    parts.append(
        f'<div style="{_card_style}">'
        f'<div style="{_header_style}">'
        f'<div style="font-size:15px; font-weight:bold; margin-bottom:4px;">{_html.escape(item_id)} — {_html.escape(domain)}</div>'
        f'<div style="font-size:13px; color:#a0a0a0;">Prompt: {_html.escape(prompt_text)}</div>'
        f"</div>"
        f'<table style="width:100%; border-collapse:collapse;">{"".join(rows)}</table>'
        f"</div>"
    )

display(HTML("\n".join(parts)))

In [ ]:
"""Cell 17: Quantitative analysis — length, token overlap with baseline, uniqueness."""

from collections import Counter

def jaccard_tokens(a: str, b: str) -> float:
    """Token-level Jaccard similarity."""
    sa = set(a.lower().split())
    sb = set(b.lower().split())
    if not sa and not sb:
        return 1.0
    return len(sa & sb) / len(sa | sb) if (sa | sb) else 0.0


quant_rows = []
for item_id in results_df["id"].unique():
    sub = results_df[results_df["id"] == item_id].sort_values("alpha")
    baseline = sub[sub["alpha"] == 0].iloc[0]["reply"]
    for _, row in sub.iterrows():
        reply = row["reply"]
        quant_rows.append({
            "id": item_id,
            "alpha": row["alpha"],
            "reply_len_words": len(reply.split()),
            "reply_len_chars": len(reply),
            "jaccard_vs_baseline": round(jaccard_tokens(reply, baseline), 3),
        })

quant_df = pd.DataFrame(quant_rows)
quant_df.to_csv(OUTPUT_DIR / "quantitative_metrics.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'quantitative_metrics.csv'}")

# Aggregate by alpha
agg = quant_df.groupby("alpha").agg(
    mean_words=("reply_len_words", "mean"),
    std_words=("reply_len_words", "std"),
    mean_jaccard=("jaccard_vs_baseline", "mean"),
    std_jaccard=("jaccard_vs_baseline", "std"),
).round(3)
print("Aggregate statistics by α:")
print(agg)
print()

# Per-item summary
pivot = quant_df.pivot(index="id", columns="alpha", values="jaccard_vs_baseline")
print("Jaccard similarity vs baseline (α=0) per item:")
print(pivot.round(3))

In [ ]:
"""Cell 18: Visualize steering effect."""

import matplotlib
matplotlib.use("Agg")  # non-interactive backend for notebooks without display
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Mean reply length vs α
ax = axes[0]
agg_plot = quant_df.groupby("alpha")["reply_len_words"].mean()
ax.plot(agg_plot.index, agg_plot.values, "o-", color="tab:blue")
ax.set_xlabel("α (steering strength)")
ax.set_ylabel("Mean reply length (words)")
ax.set_title("Reply length vs α")
ax.grid(True, alpha=0.3)

# Plot 2: Mean Jaccard similarity vs baseline
ax = axes[1]
jac_plot = quant_df.groupby("alpha")["jaccard_vs_baseline"].mean()
ax.plot(jac_plot.index, jac_plot.values, "s-", color="tab:orange")
ax.set_xlabel("α (steering strength)")
ax.set_ylabel("Jaccard similarity vs α=0")
ax.set_title("Token overlap with baseline")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

# Plot 3: Layer diagnostics — |v_bridge| and frac_positive
ax = axes[2]
ax.bar(stats_df["layer"], stats_df["|v_bridge|"], alpha=0.6, label="|v_bridge|", color="tab:green")
ax2 = ax.twinx()
ax2.plot(stats_df["layer"], stats_df["frac_positive"], "D-", color="tab:red", label="frac_positive")
ax.set_xlabel("Layer")
ax.set_ylabel("|v_bridge|", color="tab:green")
ax2.set_ylabel("frac BC > D", color="tab:red")
ax.set_title("Bridge direction strength by layer")
ax.axvline(STEER_LAYER, color="black", linestyle="--", alpha=0.5, label=f"selected L={STEER_LAYER}")
ax.legend(loc="upper left", fontsize=8)
ax2.legend(loc="upper right", fontsize=8)

plt.tight_layout()
OUTPUT_DIR.mkdir(exist_ok=True)
plot_fname = OUTPUT_DIR / f"bridge_steering_analysis_{TASK_TYPE.value}.png"
plt.savefig(plot_fname, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_fname}")

## 6. Save results

Save all outputs and the steering vector for later analysis or follow-up experiments.

In [12]:
"""Cell 22: Save everything."""

OUTPUT_DIR.mkdir(exist_ok=True)

# 1. Steering vector
torch.save({
    "v_bridge": {L: v_bridge[L] for L in PROBE_LAYERS},
    "centroids_D": {L: centroids_D[L] for L in PROBE_LAYERS},
    "centroids_B": {L: centroids_B[L] for L in PROBE_LAYERS},
    "steer_layer": STEER_LAYER,
    "v_steer": v_steer,
    "v_steer_unit": v_steer / (v_steer.norm() + 1e-8),
    "model_name": MODEL_NAME,
    "task_type": TASK_TYPE.value,
    "probe_layers": PROBE_LAYERS,
    "n_items": len(pairs),
}, OUTPUT_DIR / "steering_vectors.pt")

# 2. Layer diagnostics
stats_df.to_csv(OUTPUT_DIR / "layer_diagnostics.csv", index=False)

# 3. Ablation (injection modes) — if you ran Cell 20
_ablation_df = globals().get("ablation_df")
if _ablation_df is not None:
    _ablation_df.to_json(OUTPUT_DIR / "ablation_results.json", orient="records", indent=2)
    print(f"Saved: {OUTPUT_DIR / 'ablation_results.json'}")

print(f"Run outputs in {OUTPUT_DIR}/")
print("Files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

Run outputs in bridge_steering_outputs_aut_20260223_072806/
Files:
  alpha_sweep_results.json  (516.2 KB)
  fewshot_results.csv  (401.9 KB)
  fewshot_results.json  (432.8 KB)
  layer_diagnostics.csv  (0.8 KB)
  pe_results.csv  (131.2 KB)
  pe_results.json  (136.9 KB)
  pe_results_aut.csv  (125.9 KB)
  pe_results_aut.json  (132.1 KB)
  steering_alpha_sweep.csv  (807.6 KB)
  steering_baseline.csv  (76.9 KB)
  steering_vectors.pt  (255.2 KB)
  twohop_results.csv  (206.4 KB)
  twohop_results.json  (219.7 KB)
  twohop_results_aut.csv  (195.1 KB)
  twohop_results_aut.json  (209.5 KB)
